# 这是Jerry自用的产品开发说明书，如有必要可以给GPT或者Claude浏览

---

# 表01，store_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 01 · store_list — Store master table
-- 文件 01 · store_list — 门店主表
-- =============================================
-- Dependencies / 依赖:
--   (none — this is the foundational table, created first)
--   （无 — 基础表，最先创建）
-- Dependents / 被依赖:
--   Almost all other tables reference this table via store_id FK
--   几乎所有其他表通过 store_id FK 引用本表
-- =============================================

CREATE TABLE IF NOT EXISTS public.store_list (

  -- Store ID: business-defined text identifier (e.g., "decarie", "marcel", "parcex")
  -- 门店 ID：业务端自定义的文本标记（如 "decarie"、"marcel"、"parcex"）
  store_id text PRIMARY KEY,

  -- Store display name (required)
  -- 门店名称（必填）
  store_name text NOT NULL,

  -- Store phone number (optional)
  -- 门店电话（可选）
  store_phone_number text DEFAULT NULL,

  -- Store physical address (optional)
  -- 门店地址（可选）
  store_address text DEFAULT NULL,

  -- Store email address, used for billing / settlement
  -- 门店邮箱，用于对账结算
  store_email_address text DEFAULT NULL,

  -- Email address for receiving automated business reports
  -- 报表接收邮箱，用于自动发送经营报表等
  store_report_email text DEFAULT NULL,

  -- Store sales policy printed on sales receipts
  -- 门店销售政策（打印在销售小票上）
  store_sale_policy text DEFAULT NULL,

  -- Store repair policy printed on repair tickets / repair receipts
  -- 门店维修政策（打印在维修工单/维修收据上）
  store_repair_policy text DEFAULT NULL,

  -- Store website URL
  -- 门店网站地址
  store_website text DEFAULT NULL,

  -- Store postcode / ZIP code
  -- 门店邮编
  store_postcode text DEFAULT NULL,

  -- Google review QR reference (recommended: URL or storage path to QR image)
  -- Google Review 二维码引用（推荐存 URL 或二维码图片存储路径）
  store_qr text DEFAULT NULL,

  -- Text displayed before the store QR on printouts
  -- 打印时显示在门店二维码前的一段话
  store_qr_note_before text DEFAULT NULL,

  -- Text displayed after the store QR on printouts
  -- 打印时显示在门店二维码后的一句话
  store_qr_note_after text DEFAULT NULL,

  -- Tax rate configuration stored as a JSONB object
  -- e.g. {"GST": 0.05, "QST": 0.09975}
  -- The CHECK constraint below ensures this is always a JSON object (not array, string, etc.)
  -- 税率配置，以 JSONB 对象存储
  -- 例如 {"GST": 0.05, "QST": 0.09975}
  -- 下方 CHECK 约束确保必须是 JSON 对象类型（不能是数组、字符串等）
  tax_rates jsonb NOT NULL DEFAULT '{}'::jsonb,

  -- File path to store logo image (e.g., "stores/decarie/logo.png")
  -- 门店 Logo 图片的存储路径（如 "stores/decarie/logo.png"）
  store_logo text DEFAULT NULL,

  -- Soft delete timestamp; NULL = active store, non-NULL = closed / disabled
  -- 软删除时间戳；NULL 表示正常营业，非 NULL 表示门店已关闭/停用
  deleted_at timestamptz DEFAULT NULL,

  -- CHECK: tax_rates must be a JSON object (not array, string, number, etc.)
  -- 约束：tax_rates 必须是 JSON 对象类型
  CONSTRAINT chk_store_list_tax_rates_is_object
    CHECK (jsonb_typeof(tax_rates) = 'object')
);


-- =============================================
-- Migration safety patch: add newly introduced store profile fields
-- 迁移兼容补丁：补充新增门店信息字段
-- =============================================
ALTER TABLE public.store_list ADD COLUMN IF NOT EXISTS store_sale_policy text;
ALTER TABLE public.store_list ADD COLUMN IF NOT EXISTS store_repair_policy text;
ALTER TABLE public.store_list ADD COLUMN IF NOT EXISTS store_website text;
ALTER TABLE public.store_list ADD COLUMN IF NOT EXISTS store_postcode text;
ALTER TABLE public.store_list ADD COLUMN IF NOT EXISTS store_qr text;
ALTER TABLE public.store_list ADD COLUMN IF NOT EXISTS store_qr_note_before text;
ALTER TABLE public.store_list ADD COLUMN IF NOT EXISTS store_qr_note_after text;


# 表02，user_rights_templates

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 02 · user_rights_templates — Permission template table
-- 文件 02 · user_rights_templates — 用户权限模板表
-- =============================================
-- Dependencies / 依赖:
--   (none)
--   （无）
-- Dependents / 被依赖:
--   04_store_user_rights (template_id FK)
-- =============================================
-- Defines reusable permission templates for store employees.
-- Each template is a combination of boolean permission flags.
-- Templates are assigned to users per-store via the store_user_rights junction table.
-- Templates do NOT use soft delete: before removing a template, all users
-- referencing it must be migrated to another template first.
-- The FK constraint will block deletion if any user still references the template.
-- ─────────────────────────────────────────────
-- 定义门店员工的权限组合模板。
-- 每个模板是一组布尔权限标志的组合。
-- 通过 store_user_rights 关联表将模板分配给具体门店的具体用户。
-- 模板不使用软删除：停用模板前需先将所有引用该模板的用户迁移至其他模板，
-- 外键约束会阻止在仍有用户引用时直接删除模板，这是预期行为。
-- =============================================

CREATE TABLE IF NOT EXISTS public.user_rights_templates (

  -- Auto-increment primary key, starting from 0; should not be manually assigned
  -- 模板自增主键，从 0 开始；业务端不应手动指定
  template_id integer
    GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0)
    PRIMARY KEY,

  -- Template display name, globally unique; used in UI for identification
  -- 模板名称，全局唯一；用于 UI 展示和识别
  template_name text NOT NULL,

  -- Optional description / notes about this template
  -- 模板描述/备注（可选）
  description text DEFAULT NULL,

  -- =============================================
  -- Permission flags (boolean, all default to false)
  -- 权限字段（布尔值，默认均为 false）
  -- =============================================

  -- Can view business reports (sales summaries, daily totals, profit reports, etc.)
  -- 是否可以查看经营报表（销售汇总、日结、利润报表等）
  can_view_report boolean NOT NULL DEFAULT false,

  -- Can modify store settings (store info, tax rates, receipt layout, etc.)
  -- 是否可以修改门店设置（门店信息、税率、收据样式等）
  can_edit_settings boolean NOT NULL DEFAULT false,

  -- Can create / edit invoices, quotes, and modification orders
  -- 是否可以编辑/创建发票（含报价单、修改单等）
  can_edit_invoice boolean NOT NULL DEFAULT false,

  -- Can manage other employees (assign permissions, edit employee profiles, etc.)
  -- 是否可以管理其他员工（设置权限、修改员工信息等）
  can_manage_user boolean NOT NULL DEFAULT false,

  -- Can use the stocktake feature to manually adjust inventory quantities
  -- 是否可以使用盘点功能手动调整库存数量
  can_adjust_inventory boolean NOT NULL DEFAULT false,

  -- Whether this is a "full-access info" user:
  --   true  = UI displays all real business data (cost, profit, margins, etc.)
  --   false = UI hides or masks sensitive data, showing substitute content
  -- 是否为"真实信息"用户：
  --   true  = UI 正常显示所有真实业务数据（成本、利润等）
  --   false = UI 层面隐藏或脱敏信息，展示替代内容
  is_true_user boolean NOT NULL DEFAULT true,

  -- UNIQUE: template name must be globally unique
  -- 唯一约束：模板名称必须全局唯一
  CONSTRAINT uq_user_rights_templates_name UNIQUE (template_name)
);


# 表03，user_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 03 · user_list — Employee / user master table
-- 文件 03 · user_list — 员工/用户主表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list (created_store_id FK)
-- Dependents / 被依赖:
--   04_store_user_rights, 10_batch_list, 11_shift_list,
--   12_transaction_list, 14_store_inventory_adjustment_list,
--   16_serialized_event_list, 17_store_transfer_list,
--   19_store_item_history_list, 20_purchase_order_list,
--   22_repair_ticket_list
-- Shared components created here / 本文件创建的共享组件:
--   Function public.set_updated_at() — used by ALL mutable tables in files
--   04, 05, 06, 07, 08, 09, 10, 11, 12, 17, 20, 21, 22, 23
-- =============================================
-- Self-managed authentication system. The password field stores
-- bcrypt/argon2 hashes; the client hashes the password before sending.
-- Supabase Auth is NOT used.
-- ─────────────────────────────────────────────
-- 自管登录体系。密码字段存储 bcrypt/argon2 哈希值，
-- 客户端在发送前对密码进行哈希处理。不使用 Supabase Auth。
-- =============================================

-- =============================================
-- Shared function: set_updated_at()
-- 共享函数：set_updated_at()
-- =============================================
-- Automatically refreshes the updated_at column to now() on every UPDATE.
-- Any table that binds this trigger will have its updated_at auto-maintained.
-- >>> This function is reused by ALL subsequent mutable tables <<<
-- ─────────────────────────────────────────────
-- 通用 updated_at 自动刷新函数：
-- 任何绑定本触发器的表在 UPDATE 时自动将 updated_at 刷新为当前时间戳。
-- >>> 本函数被所有后续可变表复用 <<<
-- =============================================
CREATE OR REPLACE FUNCTION public.set_updated_at()
RETURNS trigger
LANGUAGE plpgsql
AS $$
BEGIN
  NEW.updated_at = now();
  RETURN NEW;
END;
$$;

-- =============================================
-- Table creation / 建表
-- =============================================
CREATE TABLE IF NOT EXISTS public.user_list (

  -- Auto-increment primary key, starting from 0; should not be manually assigned
  -- 用户全局自增主键，从 0 开始；业务端不应手动指定
  user_id integer
    GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0)
    PRIMARY KEY,

  -- Login username; unique among active (non-deleted) users via partial unique index below.
  -- After soft delete, the username is released and can be reused by a new user.
  -- 登录用户名；在活跃（未删除）用户中唯一（见下方 partial unique index）。
  -- 软删除后用户名释放，新用户可复用。
  user_name text NOT NULL,

  -- Password hash (bcrypt / argon2); may be NULL for SSO login or initial setup
  -- 密码哈希（bcrypt / argon2）；SSO 登录或初始未设密码时可为空
  user_password text DEFAULT NULL,

  -- Optional description / notes about this user
  -- 用户备注描述（可选）
  description text DEFAULT NULL,

  -- User's phone number (optional)
  -- 用户手机号（可选）
  user_phone_number text DEFAULT NULL,

  -- User's email address (optional)
  -- 用户邮箱（可选）
  user_email_address text DEFAULT NULL,

  -- Record creation timestamp
  -- 记录创建时间
  created_at timestamptz NOT NULL DEFAULT now(),

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- Store that created this user record
  -- 创建该用户记录的门店 ID
  created_store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Soft delete timestamp; NULL = active, non-NULL = disabled / deactivated
  -- 软删除时间戳；NULL 表示正常，非 NULL 表示已停用
  deleted_at timestamptz DEFAULT NULL,

  -- CHECK: username must not be blank (empty or whitespace-only)
  -- 约束：用户名不能为空白字符串
  CONSTRAINT chk_user_list_user_name_not_blank
    CHECK (length(btrim(user_name)) > 0)
);

-- =============================================
-- Partial unique index: username must be unique among active (non-deleted) users.
-- After soft delete the username is released, allowing a new user to reuse it.
-- ─────────────────────────────────────────────
-- 部分唯一索引：用户名在活跃（未删除）用户中唯一。
-- 软删除后用户名释放，可被新用户使用。
-- =============================================
CREATE UNIQUE INDEX IF NOT EXISTS uq_user_list_user_name_active
  ON public.user_list (user_name)
  WHERE deleted_at IS NULL;

-- =============================================
-- Trigger: auto-refresh updated_at on every UPDATE (reuses set_updated_at())
-- 触发器：每次 UPDATE 自动刷新 updated_at（复用 set_updated_at()）
-- =============================================
DROP TRIGGER IF EXISTS trg_user_list_updated_at ON public.user_list;
CREATE TRIGGER trg_user_list_updated_at
BEFORE UPDATE ON public.user_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- =============================================
-- Index: quickly find active (non-deleted) users
-- 索引：快速查找活跃（未删除）用户
-- =============================================
CREATE INDEX IF NOT EXISTS idx_user_list_active
  ON public.user_list (user_id)
  WHERE deleted_at IS NULL;


# 表04，store_user_rights，

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 04 · store_user_rights — Store-User-Permission junction table
-- 文件 04 · store_user_rights — 门店-用户-权限关联表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list  (store_id FK)
--   02_user_rights_templates (template_id FK)
--   03_user_list   (user_id FK)
-- Dependents / 被依赖:
--   (none)
--   （无）
-- =============================================
-- Junction table that links a user to a permission template within a specific store.
-- PK = (user_id, store_id), meaning each user can have exactly ONE permission
-- template per store (but may have different templates in different stores).
-- All three FK columns have constraints, ensuring:
--   - Cannot reference a non-existent user, store, or permission template
--   - Must migrate all users off a template before deleting it (FK blocks deletion)
--   - Must clean up junction records before deleting a store
-- ─────────────────────────────────────────────
-- 关联表：将用户与特定门店内的权限模板关联。
-- 主键 = (user_id, store_id)，即每个用户在每个门店只能有一个权限模板
-- （但在不同门店可以有不同模板）。
-- 三个外键列均有约束，保证：
--   - 不能引用不存在的用户、门店或权限模板
--   - 删除权限模板前需先迁移所有关联用户（外键拦截）
--   - 删除门店前需先清理关联记录
-- =============================================

CREATE TABLE IF NOT EXISTS public.store_user_rights (

  -- User ID, FK to user_list
  -- 用户 ID，外键指向 user_list
  user_id integer NOT NULL REFERENCES public.user_list(user_id),

  -- Store ID, FK to store_list
  -- 门店 ID，外键指向 store_list
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Permission template ID, FK to user_rights_templates
  -- 权限模板 ID，外键指向 user_rights_templates
  template_id integer NOT NULL REFERENCES public.user_rights_templates(template_id),

  -- Record creation timestamp
  -- 记录创建时间
  created_at timestamptz NOT NULL DEFAULT now(),

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- Composite PK: one user can have only one permission template per store
  -- 联合主键：同一用户在同一门店只能有一个权限模板
  CONSTRAINT store_user_rights_pkey PRIMARY KEY (user_id, store_id)
);

-- =============================================
-- Trigger: auto-refresh updated_at on every UPDATE
-- (reuses set_updated_at() created in 03_user_list)
-- 触发器：每次 UPDATE 自动刷新 updated_at
-- （复用 03_user_list 中创建的 set_updated_at()）
-- =============================================
DROP TRIGGER IF EXISTS trg_store_user_rights_updated_at ON public.store_user_rights;
CREATE TRIGGER trg_store_user_rights_updated_at
BEFORE UPDATE ON public.store_user_rights
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- =============================================
-- Index: look up all user permissions for a given store
-- 索引：查询某门店下所有用户的权限
-- =============================================
CREATE INDEX IF NOT EXISTS idx_store_user_rights_store
  ON public.store_user_rights (store_id);

-- =============================================
-- Index: find which users reference a given template (useful before deleting a template)
-- 索引：查询某权限模板被哪些用户使用（删除模板前使用）
-- =============================================
CREATE INDEX IF NOT EXISTS idx_store_user_rights_template
  ON public.store_user_rights (template_id);


# 表05，supplier_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 05 · supplier_list — Supplier master table
-- 文件 05 · supplier_list — 供应商主表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list (created_store_id FK)
-- Dependents / 被依赖:
--   06_mother_inventory_list (supplier_id FK)
--   20_purchase_order_list   (supplier_id FK)
-- =============================================

CREATE TABLE IF NOT EXISTS public.supplier_list (

  -- Auto-increment primary key, starting from 0; should not be manually assigned
  -- 全局自增主键，从 0 开始；业务端不应手动指定
  supplier_id integer
    GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0)
    PRIMARY KEY,

  -- Supplier name; unique among active (non-deleted) records via partial unique index below
  -- 供应商名称；在活跃（未删除）记录中唯一（见下方 partial unique index）
  supplier_name text NOT NULL,

  -- Supplier phone number (optional)
  -- 供应商电话（可选）
  supplier_phone_number text DEFAULT NULL,

  -- Supplier email address (optional)
  -- 供应商邮箱（可选）
  supplier_email_address text DEFAULT NULL,

  -- Store that created this supplier record
  -- 创建该供应商记录的门店 ID
  created_store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Record creation timestamp
  -- 记录创建时间
  created_at timestamptz NOT NULL DEFAULT now(),

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- Soft delete timestamp; NULL = active, non-NULL = disabled
  -- 软删除时间戳；NULL 表示正常，非 NULL 表示已停用
  deleted_at timestamptz DEFAULT NULL
);

-- =============================================
-- Partial unique index: supplier name must be unique among active records.
-- After soft delete the name is released, allowing reuse.
-- ─────────────────────────────────────────────
-- 部分唯一索引：供应商名称在活跃记录中唯一。
-- 软删除后名称释放，可被新记录使用。
-- =============================================
CREATE UNIQUE INDEX IF NOT EXISTS uq_supplier_list_name_active
  ON public.supplier_list (supplier_name)
  WHERE deleted_at IS NULL;

-- =============================================
-- Trigger: auto-refresh updated_at on every UPDATE
-- (reuses set_updated_at() created in 03_user_list)
-- 触发器：每次 UPDATE 自动刷新 updated_at
-- （复用 03_user_list 中创建的 set_updated_at()）
-- =============================================
DROP TRIGGER IF EXISTS trg_supplier_list_updated_at ON public.supplier_list;
CREATE TRIGGER trg_supplier_list_updated_at
BEFORE UPDATE ON public.supplier_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- =============================================
-- Index: quickly find active (non-deleted) suppliers
-- 索引：快速查找活跃（未删除）供应商
-- =============================================
CREATE INDEX IF NOT EXISTS idx_supplier_list_active
  ON public.supplier_list (supplier_id)
  WHERE deleted_at IS NULL;


# 表06，mother_inventory_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 06 · mother_inventory_list — Global product catalog
-- 文件 06 · mother_inventory_list — 全局商品母表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list    (created_store_id FK)
--   05_supplier_list (supplier_id FK)
-- Dependents / 被依赖:
--   08_store_inventory_list, 09_store_serialized_list,
--   13_transaction_line_list, 15_adjustment_line_list,
--   18_store_transfer_line_list, 19_store_item_history_list,
--   21_purchase_order_line_list, 23_repair_ticket_line_list
-- Shared components created here / 本文件创建的共享组件:
--   ENUM  public.inventory_mode  — also used by 08, 09, 13, 15, 18, 21
--   ENUM  public.valuation_method — this table only
--   Function public.assign_variant_id_per_item() — this table only
-- =============================================

-- =============================================
-- ENUM: inventory_mode — Inventory tracking mode
-- 枚举：inventory_mode — 库存跟踪模式
-- =============================================
-- >>> Referenced by trigger functions in files 08, 09, 13, 15, 18, 21 <<<
-- >>> 被以下文件的触发器函数引用：08, 09, 13, 15, 18, 21 <<<
--
--   service    = Service items mainly include unit voucher, charging port adjustment, virus cleaning, shipping, etc.
--              = 服务类商品主要包括：unit voucher、charging port adjustment、virus cleaning、shipping 等。
--   untracked  = Untracked items include unpackaged data cables, plugs/adapters, phone cases, screen protectors, and misc items.
--              = untracked 包括：无包装数据线、插头/适配器、手机壳、手机膜，以及 misc item。
--   tracked    = Tracked items are mostly packaged goods, including power banks, car mounts, etc.
--              = tracked 主要是绝大部分有包装的商品，包括充电宝、car mount 等。
--   serialized = Per-unit tracking with unique serial numbers (e.g., phones with IMEI)
--              = 序列号管理跟踪，每件唯一（如手机 IMEI）
DO $$
BEGIN
  CREATE TYPE public.inventory_mode AS ENUM ('service', 'untracked', 'tracked', 'serialized');
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =============================================
-- ENUM: valuation_method — Product cost valuation method
-- 枚举：valuation_method — 商品成本估值方式
-- =============================================
--   average = Weighted moving average cost (auto-updated on each purchase)
--           = 加权滑动平均成本（每次进货自动更新）
--   rate    = Cost calculated by fixed ratio (e.g., purchase price × coefficient)
--           = 按固定比率计算（如进价 × 系数）
--   fixed   = Fixed cost, manually specified, never auto-updated
--           = 固定成本，手动指定，不自动更新
DO $$
BEGIN
  CREATE TYPE public.valuation_method AS ENUM ('average', 'rate', 'fixed');
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =============================================
-- Function: assign_variant_id_per_item()
-- 函数：assign_variant_id_per_item()
-- =============================================
-- Auto-assigns variant_id per item_id (0, 1, 2, ...) on INSERT.
-- Uses pg_advisory_xact_lock to prevent race conditions during concurrent inserts.
-- Note: Uses COUNT(*) as the next variant_id — if item_id already has N rows
-- (including soft-deleted), the new row gets variant_id = N.
-- Warning: Soft-deleted rows are still counted, so gaps may appear in the sequence
-- (e.g., variant_id=1 is soft-deleted but the number is NOT reused).
-- ─────────────────────────────────────────────
-- 按 item_id 自动分配 variant_id（0, 1, 2, ...），在 INSERT 时触发。
-- 使用 pg_advisory_xact_lock 防止并发插入时产生重复编号。
-- 注意：使用 COUNT(*) 作为下一个 variant_id——若该 item_id 已有 N 条记录
-- （含软删除），新记录的 variant_id = N。
-- 警告：软删除的记录仍会被计入，因此序列中可能出现"空洞"
-- （例如 variant_id=1 已软删除，但序号不会被复用）。
-- ─────────────────────────────────────────────
-- Advisory Lock Namespace Allocation / 咨询锁命名空间分配:
--   1001 = assign_variant_id_per_item()         (this file, 06)
--   1002 = assign_batch_id_per_store()           (file 10)
--   1003 = assign_shift_id_per_store()           (file 11)
--   1004 = assign_purchase_order_id_per_store()  (file 20)
--   1010 = assign_demand_id_per_store()          (file 24)
-- =============================================
CREATE OR REPLACE FUNCTION public.assign_variant_id_per_item()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  -- Variable legend / 变量说明:
  --   next_vid = next variant_id to assign for this item_id / 本 item_id 下要分配的下一个 variant_id
  next_vid int;
BEGIN
  -- Guard: item_id must not be null / 防御性检查：item_id 不能为空
  IF NEW.item_id IS NULL THEN
    RAISE EXCEPTION 'item_id cannot be null';
  END IF;

  -- Serialize inserts for the same item_id using advisory lock (namespace 1001)
  -- 对同一 item_id 的插入串行化，使用咨询锁（命名空间 1001）
  PERFORM pg_advisory_xact_lock(1001, NEW.item_id);

  SELECT COUNT(*)::int
    INTO next_vid
  FROM public.mother_inventory_list
  WHERE item_id = NEW.item_id;  -- includes soft-deleted rows / 含软删除行

  NEW.variant_id := next_vid;
  RETURN NEW;
END;
$$;

-- =============================================
-- Table creation / 建表
-- =============================================
CREATE TABLE IF NOT EXISTS public.mother_inventory_list (

  -- Global unique PK, auto-generated IDENTITY starting from 0
  -- Business logic should NOT use this value directly as a display ID
  -- 全局唯一主键，IDENTITY 自动生成，从 0 开始递增
  -- 业务端不应直接使用本值做展示 ID
  unique_id integer GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0) PRIMARY KEY,

  -- Product number (item_id): different variants of the same product share the same item_id
  -- Valid range: 100000–999999 (enforced by CHECK constraint below)
  -- 商品编号（item_id）：同一商品的不同规格共享同一 item_id
  -- 有效范围：100000–999999（由下方 CHECK 约束强制）
  item_id integer NOT NULL,

  -- Variant number: auto-assigned per item_id starting from 0 (0 = default/only variant)
  -- 规格编号：同一 item_id 下从 0 开始自动分配（0 = 默认/唯一规格）
  variant_id integer NOT NULL DEFAULT 0,

  -- Product barcodes (UPC): supports multiple barcodes per product; may be empty or duplicated across products
  -- 商品条码（UPC）：支持多条码绑定；允许为空，允许与其他条目重复
  item_upc text[] NOT NULL DEFAULT '{}'::text[],

  -- Product display name (required)
  -- 商品名称（必填）
  item_name text NOT NULL,

  -- Hierarchical category path as a text array; empty array = "uncategorized"
  -- Array elements represent levels from root category to subcategory
  -- e.g., {"Electronics", "Phone Accessories", "Cases"}
  -- 树形分类路径，以文本数组存储；空数组 = "未分类"
  -- 数组元素按顺序表示从根分类到子分类
  -- 例如 {"Electronics", "Phone Accessories", "Cases"}
  category_path text[] NOT NULL DEFAULT '{}'::text[],

  -- Product description / notes (optional)
  -- 商品描述/备注（可选）
  description text DEFAULT NULL,

  -- Manufacturer / brand name (optional)
  -- 商品生产商/品牌（可选）
  manufacture text DEFAULT NULL,

  -- Supplier FK (replaces the old text-based supplier column)
  -- 供应商外键（替代旧的文本类型 supplier 字段）
  supplier_id integer DEFAULT NULL REFERENCES public.supplier_list(supplier_id),

  -- Compatible device models; mainly for repair parts
  -- Stores device model strings this part is compatible with
  -- e.g., {"iPhone 15 Pro", "iPhone 15 Pro Max"}
  -- 适配设备型号列表；主要用于维修配件
  -- 以数组形式记录该配件兼容的设备型号字符串
  -- 例如 {"iPhone 15 Pro", "iPhone 15 Pro Max"}
  device_compatibility text[] NOT NULL DEFAULT '{}'::text[],

  -- Inventory tracking mode; see inventory_mode ENUM definition above
  -- 库存跟踪模式；参见上方 inventory_mode 枚举定义
  inventory_mode public.inventory_mode NOT NULL,

  -- Cost valuation method; see valuation_method ENUM above; defaults to weighted moving average
  -- 成本估值方式；参见上方 valuation_method 枚举定义；默认使用加权滑动平均
  valuation_method public.valuation_method NOT NULL DEFAULT 'average',

  -- Default cost per unit (inherited by store tables on INSERT)
  -- 默认单位成本（门店表 INSERT 时继承此值）
  default_cost numeric(10,2),

  -- Default retail price per unit (inherited by store tables on INSERT)
  -- 默认零售价格（门店表 INSERT 时继承此值）
  default_price numeric(10,2),

  -- Record creation timestamp
  -- 记录创建时间
  created_at timestamptz NOT NULL DEFAULT now(),

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- Store that created this product record
  -- 创建该商品记录的门店 ID
  created_store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Soft delete timestamp; NULL = active, non-NULL = removed from catalog
  -- 软删除时间戳；NULL 表示正常，非 NULL 表示已从目录中移除
  deleted_at timestamptz DEFAULT NULL,

  -- CHECK: item_id must be in the range 100000–999999
  -- 约束：item_id 必须在 100000–999999 范围内
  CONSTRAINT chk_mother_inventory_list_item_id_range
    CHECK (item_id BETWEEN 100000 AND 999999)
);

-- =============================================
-- Unique index: item_name must be globally unique among active rows
-- 唯一索引：item_name 在全局活跃记录中不可重复
-- Note: this prevents exact same-name duplicates; different naming systems can still coexist by using different names.
-- 说明：该约束仅禁止“同名”重复；不同命名体系（不同名称）仍可并存。
CREATE UNIQUE INDEX IF NOT EXISTS uq_mother_item_name_active
  ON public.mother_inventory_list (item_name)
  WHERE deleted_at IS NULL;

-- Unique index: (item_id, variant_id) must be globally unique (including soft-deleted rows)
-- 唯一索引：(item_id, variant_id) 必须全局唯一（包含软删除行）
-- =============================================
CREATE UNIQUE INDEX IF NOT EXISTS uq_mother_item_variant
ON public.mother_inventory_list (item_id, variant_id);

-- =============================================
-- Triggers / 触发器
-- =============================================

-- Trigger: auto-refresh updated_at (reuses set_updated_at() from file 03)
-- 触发器：自动刷新 updated_at（复用 03_user_list 中的 set_updated_at()）
DROP TRIGGER IF EXISTS trg_mother_inventory_list_updated_at ON public.mother_inventory_list;
CREATE TRIGGER trg_mother_inventory_list_updated_at
BEFORE UPDATE ON public.mother_inventory_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- Trigger: auto-assign variant_id on INSERT
-- 触发器：INSERT 时自动分配 variant_id
DROP TRIGGER IF EXISTS trg_mother_inventory_list_assign_variant_id ON public.mother_inventory_list;
CREATE TRIGGER trg_mother_inventory_list_assign_variant_id
BEFORE INSERT ON public.mother_inventory_list
FOR EACH ROW
EXECUTE FUNCTION public.assign_variant_id_per_item();

-- =============================================
-- Indexes / 索引
-- =============================================

-- Look up products by item_id (active only)
-- 按 item_id 查找商品（仅活跃记录）
CREATE INDEX IF NOT EXISTS idx_mother_inventory_list_item_id
  ON public.mother_inventory_list (item_id)
  WHERE deleted_at IS NULL;

-- Filter products by inventory_mode
-- 按库存跟踪模式筛选商品
CREATE INDEX IF NOT EXISTS idx_mother_inventory_list_inventory_mode
  ON public.mother_inventory_list (inventory_mode);

-- GIN index for category_path array searches (e.g., find all products in a category)
-- GIN 索引：用于分类路径数组搜索（如查找某分类下的所有商品）
CREATE INDEX IF NOT EXISTS gin_mother_inventory_list_category_path
  ON public.mother_inventory_list USING gin (category_path);

-- GIN index for device_compatibility array searches (e.g., find parts for a specific phone model)
-- GIN 索引：用于适配设备数组搜索（如查找适配某手机型号的配件）
CREATE INDEX IF NOT EXISTS gin_mother_inventory_list_device_compatibility
  ON public.mother_inventory_list USING gin (device_compatibility);

-- GIN index for item_upc barcode array searches (e.g., scan a barcode to find the product)
-- GIN 索引：用于商品条码数组搜索（如扫码查找商品）
CREATE INDEX IF NOT EXISTS gin_mother_inventory_list_item_upc
  ON public.mother_inventory_list USING gin (item_upc);


# 表07，customer_list

```sql
-- Async requirement: YES - offline POS must continue high-frequency essential operations using local snapshot; sync changes to cloud after reconnection.
-- 异步需求：是 - POS 离线时需依赖本地快照继续高频必要操作，网络恢复后将变更同步到云端。
-- =============================================
-- File 07 · customer_list — Customer master table
-- 文件 07 · customer_list — 客户主表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list (created_store_id FK)
-- Dependents / 被依赖:
--   12_transaction_list   (customer_id FK)
--   22_repair_ticket_list (customer_id FK)
-- =============================================
-- Supports offline creation: PK is a UUID v7 generated client-side.
-- When offline, the client generates the UUID locally; it syncs to Supabase when online.
-- ─────────────────────────────────────────────
-- 支持离线创建：主键由客户端生成 UUID v7。
-- 离线时客户端自行生成，联网后同步写入 Supabase。
-- =============================================

CREATE TABLE IF NOT EXISTS public.customer_list (

  -- UUID v7 primary key, generated client-side for offline support
  -- Printed on receipts as a unique customer reference
  -- UUID v7 主键，客户端生成，支持离线创建
  -- 小票上会打印本 ID 作为客户唯一凭证
  customer_id uuid PRIMARY KEY,

  -- Customer name; may be NULL for anonymous customers or to be filled in later
  -- 客户姓名；匿名客户或稍后补填时可为空
  customer_name text DEFAULT NULL,

  -- Optional description / notes about this customer
  -- 客户备注描述（可选）
  description text DEFAULT NULL,

  -- Phone number stored as text; NULL = unknown
  -- Unique among active customers via partial unique index (see below)
  -- After soft delete, the phone number is released for reuse by a new customer
  -- Offline conflict scenario: two stores create a customer with the same phone number offline;
  -- on sync, the second record hits the unique constraint — app layer handles conflict (merge or discard)
  -- 手机号码，以文本存储；NULL 表示未知
  -- 在活跃客户中唯一（见下方 partial unique index）
  -- 软删除后号码释放，可被新客户使用
  -- 离线冲突场景：两个门店离线用同一手机号建客户，同步时第二条会被唯一约束拦截，
  -- 应用层需处理冲突逻辑（提示合并或放弃）
  phone_number text DEFAULT NULL,

  -- Cached field: total balance across all stores for this customer
  -- Updated by business logic / RPC, NOT auto-calculated by the database
  -- 缓存字段：该客户在所有门店的余额合计
  -- 由业务逻辑/RPC 更新，数据库层面不自动计算
  balance_total numeric(10,2) NOT NULL DEFAULT 0,

  -- Client-side creation timestamp (provided by the client for offline scenarios, not server time)
  -- 客户端本地创建时间（离线场景下由客户端提供，不使用服务端时间）
  created_at timestamptz NOT NULL,

  -- First sync timestamp to Supabase; NULL = not yet synced
  -- 首次同步到 Supabase 的时间；NULL 表示尚未同步
  synced_at timestamptz DEFAULT NULL,

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL,

  -- Timestamp of last business activity (transaction, modification, payment, etc.)
  -- Updated by business logic / RPC
  -- 最近一次业务活动时间（交易/修改/付款等）
  -- 由业务逻辑/RPC 更新
  last_activity_at timestamptz DEFAULT NULL,

  -- Store that created this customer record
  -- 创建该客户记录的门店 ID
  created_store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Soft delete timestamp; NULL = active, non-NULL = deleted
  -- 软删除时间戳；NULL 表示正常，非 NULL 表示已删除
  deleted_at timestamptz DEFAULT NULL
);

-- =============================================
-- Trigger: auto-refresh updated_at on every UPDATE
-- (reuses set_updated_at() created in 03_user_list)
-- 触发器：每次 UPDATE 自动刷新 updated_at
-- （复用 03_user_list 中创建的 set_updated_at()）
-- =============================================
DROP TRIGGER IF EXISTS trg_customer_list_updated_at ON public.customer_list;
CREATE TRIGGER trg_customer_list_updated_at
BEFORE UPDATE ON public.customer_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- =============================================
-- Partial unique index: phone number must be unique among active customers with a non-NULL phone
-- Records with phone_number IS NULL are excluded (allows multiple anonymous customers)
-- After soft delete the phone number is released for reuse
-- ─────────────────────────────────────────────
-- 部分唯一索引：活跃且手机号非空的客户中，手机号唯一
-- phone_number IS NULL 的记录不参与唯一性检查（允许多个匿名客户）
-- 软删除后手机号释放，新客户可复用同一号码
-- =============================================
CREATE UNIQUE INDEX IF NOT EXISTS uq_customer_list_phone_active
  ON public.customer_list (phone_number)
  WHERE deleted_at IS NULL AND phone_number IS NOT NULL;

-- =============================================
-- Index: quickly find active customers by store
-- 索引：按门店快速查找活跃客户
-- =============================================
CREATE INDEX IF NOT EXISTS idx_customer_list_active
  ON public.customer_list (created_store_id)
  WHERE deleted_at IS NULL;


# 表08，store_inventory_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 08 · store_inventory_list — Per-store inventory table
-- 文件 08 · store_inventory_list — 门店库存明细表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list           (store_id FK)
--   06_mother_inventory_list (unique_id FK + inventory_mode ENUM)
-- Dependents / 被依赖:
--   15_store_inventory_adjustment_line_list (store_id, unique_id composite FK)
-- Shared components created here / 本文件创建的共享组件:
--   ENUM public.stock_bucket — also used by 15_adjustment_line, 19_store_item_history
-- =============================================
-- Per-store inventory for service / tracked / untracked items.
-- Serialized items have their own table: store_serialized_list (file 09).
-- ─────────────────────────────────────────────
-- 门店库存明细表：记录每个门店对 mother_inventory_list 中商品的实际库存状况。
-- 仅适用于 service / tracked / untracked 三种模式；
-- serialized 商品有独立的 store_serialized_list 表（文件 09），不在本处记录。
-- =============================================

-- =============================================
-- ENUM: stock_bucket — Fuzzy inventory level for untracked items
-- 枚举：stock_bucket — 非精确跟踪商品的模糊库存档位
-- =============================================
-- >>> Also used by 15_adjustment_line_list, 19_store_item_history_list <<<
-- >>> 也被 15_adjustment_line_list、19_store_item_history_list 引用 <<<
--
--   empty    = Out of stock / 无货
--   very_few = Very few left, near stockout / 极少，快断货
--   few      = Small quantity remaining / 少量
--   normal   = Normal stock level / 库存正常
--   too_much = Overstocked / 库存过多（积压）
DO $$
BEGIN
  CREATE TYPE public.stock_bucket AS ENUM ('empty', 'very_few', 'few', 'normal', 'too_much');
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =============================================
-- Table creation / 建表
-- =============================================
CREATE TABLE IF NOT EXISTS public.store_inventory_list (

  -- Store ID, FK to store_list
  -- 门店 ID，外键指向 store_list
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Product reference, FK to mother_inventory_list (NOT item_id, but the true PK unique_id)
  -- 商品引用，外键指向 mother_inventory_list（不是 item_id，而是真正的主键 unique_id）
  unique_id int NOT NULL REFERENCES public.mother_inventory_list(unique_id),

  -- Exact stock quantity; used ONLY for tracked mode
  -- May be negative: business allows selling below zero to avoid blocking sales due to count errors
  -- Must be NULL for service / untracked modes
  -- 精确库存数量；仅 tracked 模式使用
  -- 允许负数：业务上为避免因库存计数偏差阻碍销售，允许先销售后修正
  -- service / untracked 模式应为 NULL
  qty_on_hand int DEFAULT NULL,

  -- Fuzzy stock level; used ONLY for untracked mode
  -- Must be NULL for service / tracked modes
  -- 模糊库存档位；仅 untracked 模式使用
  -- service / tracked 模式应为 NULL
  stock_bucket public.stock_bucket DEFAULT NULL,

  -- Store-specific cost per unit
  -- On INSERT, if NULL, auto-inherited from mother table default_cost; afterwards updated independently
  -- 门店的实际单位成本
  -- INSERT 时若为空，自动从母表 default_cost 继承；之后独立更新，不随母表变化
  cost numeric(10,2) DEFAULT NULL,

  -- Store-specific retail price per unit
  -- On INSERT, if NULL, auto-inherited from mother table default_price; afterwards updated independently
  -- 门店的当前零售单价
  -- INSERT 时若为空，自动从母表 default_price 继承；之后独立更新
  price numeric(10,2) DEFAULT NULL,

  -- Minimum suggested price for sales staff; advisory only, not enforced
  -- 前台最低报价指导价：销售人员可参考作为报价下限，不做强制限制
  last_price numeric(10,2) DEFAULT NULL,

  -- Record creation timestamp
  -- 记录创建时间
  created_at timestamptz NOT NULL DEFAULT now(),

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- Soft delete timestamp; NULL = active, non-NULL = product removed from this store
  -- Note: before soft-deleting, ensure the mother table inventory_mode hasn't changed incompatibly.
  -- To change mode, soft-delete all store rows first, then modify the mother table.
  -- 软删除时间戳；NULL 表示正常，非 NULL 表示该门店已下架本商品
  -- 注意：软删除前应确保母表 inventory_mode 未发生不兼容的变更。
  -- 若需变更 mode，应先将所有门店的该商品行软删除，再修改母表。
  deleted_at timestamptz DEFAULT NULL,

  -- Composite PK: one row per product per store (including soft-deleted rows)
  -- 联合主键：同一门店内，同一 SKU 只能有一行（含软删除行）
  CONSTRAINT store_inventory_list_pkey PRIMARY KEY (store_id, unique_id),

  -- Mutual exclusion: qty_on_hand and stock_bucket cannot both be non-NULL
  -- (DB-level safety net; the trigger also validates this per inventory_mode)
  -- 互斥约束：qty_on_hand 和 stock_bucket 不能同时有值
  -- （数据库层面兜底；触发器也会按 inventory_mode 进行明确校验）
  CONSTRAINT chk_store_inventory_list_qty_bucket_mutex
    CHECK (NOT (qty_on_hand IS NOT NULL AND stock_bucket IS NOT NULL))
);

-- =============================================
-- Trigger A: validate inventory_mode consistency + inherit default cost/price on INSERT
-- 触发器 A：校验 inventory_mode 一致性 + INSERT 时从母表继承默认 cost/price
-- =============================================
-- Logic:
--   - On INSERT and UPDATE: validates mode consistency (soft-deleted rows skip validation)
--   - On INSERT only: inherits cost/price from mother table defaults
--   - Serialized items are rejected (must use store_serialized_list instead)
-- 逻辑说明：
--   - INSERT 和 UPDATE 时均执行 mode 一致性校验（软删除行跳过校验）
--   - 仅 INSERT 时将 cost/price 从母表默认值继承
--   - serialized 商品直接拒绝写入，应使用 store_serialized_list 表
-- =============================================
CREATE OR REPLACE FUNCTION public.store_inventory_enforce_mode_and_defaults()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  -- Variable legend / 变量说明:
  --   v_mode  = inventory_mode from mother table / 母表的库存跟踪模式
  --   m_cost  = default_cost from mother table / 母表的默认成本
  --   m_price = default_price from mother table / 母表的默认价格
  v_mode public.inventory_mode;
  m_cost numeric(10,2);
  m_price numeric(10,2);
BEGIN
  -- Soft-deleted rows skip all business validation / 软删除行跳过所有业务校验
  IF NEW.deleted_at IS NOT NULL THEN
    RETURN NEW;
  END IF;

  -- Read inventory_mode and default prices from mother table
  -- 从母表读取 inventory_mode 及默认价格
  SELECT inventory_mode, default_cost, default_price
    INTO v_mode, m_cost, m_price
  FROM public.mother_inventory_list
  WHERE unique_id = NEW.unique_id;

  IF NOT FOUND THEN
    RAISE EXCEPTION 'mother_inventory_list.unique_id % not found', NEW.unique_id;
  END IF;

  -- Serialized items must NOT be inserted here; use store_serialized_list instead
  -- serialized 商品不允许写入本表，应使用 store_serialized_list
  IF v_mode = 'serialized' THEN
    RAISE EXCEPTION 'unique_id % is serialized; use store_serialized_list table instead', NEW.unique_id;
  END IF;

  -- Service items: no inventory tracking, both qty and bucket must be NULL
  -- service 商品：不跟踪库存，qty 和 bucket 均应为 NULL
  IF v_mode = 'service' THEN
    IF NEW.qty_on_hand IS NOT NULL OR NEW.stock_bucket IS NOT NULL THEN
      RAISE EXCEPTION 'service item must have qty_on_hand NULL and stock_bucket NULL (unique_id=%)', NEW.unique_id;
    END IF;
  END IF;

  -- Tracked items: must have exact quantity, bucket must be NULL
  -- tracked 商品：应有精确数量，bucket 应为 NULL
  IF v_mode = 'tracked' THEN
    IF NEW.qty_on_hand IS NULL OR NEW.stock_bucket IS NOT NULL THEN
      RAISE EXCEPTION 'tracked item must have qty_on_hand NOT NULL and stock_bucket NULL (unique_id=%)', NEW.unique_id;
    END IF;
  END IF;

  -- Untracked items: must have bucket level, qty must be NULL
  -- untracked 商品：应有档位标记，qty 应为 NULL
  IF v_mode = 'untracked' THEN
    IF NEW.stock_bucket IS NULL OR NEW.qty_on_hand IS NOT NULL THEN
      RAISE EXCEPTION 'untracked item must have stock_bucket NOT NULL and qty_on_hand NULL (unique_id=%)', NEW.unique_id;
    END IF;
  END IF;

  -- On INSERT only: inherit default cost/price from mother table (UPDATE preserves store-specific values)
  -- 仅 INSERT 时从母表继承默认 cost/price；UPDATE 时不覆盖（门店价格独立更新）
  IF TG_OP = 'INSERT' THEN
    IF NEW.cost IS NULL THEN NEW.cost := m_cost; END IF;
    IF NEW.price IS NULL THEN NEW.price := m_price; END IF;
  END IF;

  RETURN NEW;
END;
$$;

DROP TRIGGER IF EXISTS trg_store_inventory_enforce ON public.store_inventory_list;
CREATE TRIGGER trg_store_inventory_enforce
BEFORE INSERT OR UPDATE ON public.store_inventory_list
FOR EACH ROW
EXECUTE FUNCTION public.store_inventory_enforce_mode_and_defaults();

-- =============================================
-- Trigger: auto-refresh updated_at (reuses set_updated_at() from file 03)
-- 触发器：自动刷新 updated_at（复用 03_user_list 中的 set_updated_at()）
-- =============================================
DROP TRIGGER IF EXISTS trg_store_inventory_list_updated_at ON public.store_inventory_list;
CREATE TRIGGER trg_store_inventory_list_updated_at
BEFORE UPDATE ON public.store_inventory_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- =============================================
-- Indexes / 索引
-- =============================================

-- Reverse lookup: find which stores carry a given product (active only)
-- PK already covers "list all products for a store" queries
-- 反向查询：查找哪些门店有某商品的库存（仅活跃记录）
-- PK 已覆盖"查询门店全量库存"的场景
CREATE INDEX IF NOT EXISTS idx_store_inventory_list_unique_id
  ON public.store_inventory_list (unique_id)
  WHERE deleted_at IS NULL;

-- Quick lookup of all active products in a store
-- 快速查询门店所有活跃商品
CREATE INDEX IF NOT EXISTS idx_store_inventory_list_active
  ON public.store_inventory_list (store_id)
  WHERE deleted_at IS NULL;


# 表09，store_serialized_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 09 · store_serialized_list — Serialized item inventory table
-- 文件 09 · store_serialized_list — 序列号商品库存表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list            (store_id FK)
--   06_mother_inventory_list (unique_id FK + inventory_mode ENUM)
-- Dependents / 被依赖:
--   13_transaction_line_list      (unit_id FK)
--   16_serialized_event_list      (unit_id FK)
--   18_store_transfer_line_list   (unit_id FK)
--   21_purchase_order_line_list   (unit_id FK)
-- Shared components created here / 本文件创建的共享组件:
--   ENUM public.serialized_status — also used by 16_serialized_event_list
--   Function public.mother_inventory_block_mode_change() — trigger on mother_inventory_list
-- =============================================
-- Each row represents ONE physical unit of a serialized product (e.g., a phone with an IMEI).
-- Only for products where inventory_mode = 'serialized'.
-- ─────────────────────────────────────────────
-- 每一行代表一件序列号商品的实物（如一部有 IMEI 的手机）。
-- 仅适用于 inventory_mode = 'serialized' 的商品。
-- =============================================

-- =============================================
-- ENUM: serialized_status — Lifecycle status of a serialized unit
-- 枚举：serialized_status — 序列号商品的生命周期状态
-- =============================================
-- >>> Also referenced by 16_serialized_event_list <<<
-- >>> 也被 16_serialized_event_list 引用 <<<
--
--   in_stock   = In stock, available for sale / 在库，可售
--   in_transit = Being transferred between stores / 调拨途中
--   sold       = Sold to a customer / 已售出
--   repair     = Sent out for external repair / 送外部维修中
--   lost       = Marked as lost / 标记为丢失
--   wasted     = Marked as damaged/wasted / 标记为损坏/报废
--   void       = Voided (logical deletion of the unit record) / 作废（逻辑删除该单件记录）
DO $$
BEGIN
  CREATE TYPE public.serialized_status AS ENUM ('in_stock', 'in_transit', 'sold', 'repair', 'lost', 'wasted', 'void');
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =============================================
-- Table creation / 建表
-- =============================================
CREATE TABLE IF NOT EXISTS public.store_serialized_list (

  -- Auto-increment PK for each physical unit; starting from 0
  -- 每件实物的自增主键，从 0 开始
  unit_id integer
    GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0)
    PRIMARY KEY,

  -- Store ID, FK to store_list (the store currently holding this unit)
  -- 门店 ID，外键指向 store_list（当前持有该单件的门店）
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Globally unique serial number (e.g., IMEI, SN); NEVER reused even after void/soft-delete
  -- 全局唯一序列号（如 IMEI、SN）；即使作废/软删除后仍永久占位，不可复用
  serial text NOT NULL,

  -- Product reference, FK to mother_inventory_list (must be inventory_mode = 'serialized')
  -- 商品引用，外键指向 mother_inventory_list（必须是 inventory_mode = 'serialized'）
  unique_id int NOT NULL REFERENCES public.mother_inventory_list(unique_id),

  -- Extra attributes for this unit as free-form JSONB (e.g., color, capacity, condition)
  -- Example: {"color": "Space Black", "capacity": "256GB", "condition": "A+"}
  -- 单件额外属性，自由格式 JSONB（如颜色、容量、成色等）
  -- 例如：{"color": "Space Black", "capacity": "256GB", "condition": "A+"}
  attribute jsonb DEFAULT NULL,

  -- Unit cost; on INSERT, if NULL, auto-inherited from mother table default_cost
  -- 单件成本；INSERT 时若为空，自动从母表 default_cost 继承；之后独立更新
  cost numeric(10,2) DEFAULT NULL,

  -- Unit retail price; on INSERT, if NULL, auto-inherited from mother table default_price
  -- 单件零售价；INSERT 时若为空，自动从母表 default_price 继承；之后独立更新
  price numeric(10,2) DEFAULT NULL,

  -- Minimum suggested price for sales staff; advisory only
  -- 前台最低报价指导价，仅供参考
  last_price numeric(10,2) DEFAULT NULL,

  -- Current lifecycle status of this unit (see serialized_status ENUM above)
  -- 单件当前业务状态（参见上方 serialized_status 枚举）
  status public.serialized_status NOT NULL DEFAULT 'in_stock',

  -- Record creation timestamp
  -- 记录创建时间
  created_at timestamptz NOT NULL DEFAULT now(),

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- Soft delete timestamp; NULL = active, non-NULL = deleted
  -- Note: even after soft delete, the serial is still reserved (global unique constraint has no WHERE filter)
  -- 软删除时间戳；NULL 表示正常，非 NULL 表示已删除
  -- 注意：软删除后 serial 仍占位，不可复用（全局唯一约束不带 WHERE 条件）
  deleted_at timestamptz DEFAULT NULL
);

-- =============================================
-- Serial global unique constraint (DEFERRABLE, supports in-transaction swaps)
-- Not a partial index: serial stays reserved even after soft-delete or void,
-- guaranteeing lifetime uniqueness across all records.
-- ─────────────────────────────────────────────
-- serial 全局唯一约束（DEFERRABLE，支持事务内互换）
-- 不使用 partial index：即使 deleted_at IS NOT NULL 或 status = 'void'，
-- serial 仍不可被新记录复用，保证全生命周期唯一。
-- =============================================
DO $$
BEGIN
  ALTER TABLE public.store_serialized_list
    ADD CONSTRAINT uq_store_serialized_serial
    UNIQUE (serial)
    DEFERRABLE INITIALLY IMMEDIATE;
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =============================================
-- Trigger A: validate mother table mode = 'serialized' + inherit cost/price on INSERT
-- 触发器 A：校验母表 inventory_mode = 'serialized' + INSERT 时继承 cost/price
-- =============================================
CREATE OR REPLACE FUNCTION public.store_serialized_enforce_mode()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  -- Variable legend / 变量说明:
  --   v_mode  = inventory_mode from mother table / 母表的库存跟踪模式
  --   m_cost  = default_cost from mother table / 母表的默认成本
  --   m_price = default_price from mother table / 母表的默认价格
  v_mode  public.inventory_mode;
  m_cost  numeric(10,2);
  m_price numeric(10,2);
BEGIN
  -- Read inventory_mode and default prices from mother table
  -- 从母表读取 inventory_mode 及默认价格
  SELECT inventory_mode, default_cost, default_price
    INTO v_mode, m_cost, m_price
  FROM public.mother_inventory_list
  WHERE unique_id = NEW.unique_id;

  IF NOT FOUND THEN
    RAISE EXCEPTION 'mother_inventory_list.unique_id % not found', NEW.unique_id;
  END IF;

  IF v_mode IS DISTINCT FROM 'serialized' THEN
    RAISE EXCEPTION 'unique_id % is not serialized (mode=%)', NEW.unique_id, v_mode;
  END IF;

  -- On INSERT only: inherit default cost/price (UPDATE preserves unit-specific values)
  -- 仅 INSERT 时从母表继承默认 cost/price；UPDATE 时不覆盖（单件价格独立更新）
  IF TG_OP = 'INSERT' THEN
    IF NEW.cost  IS NULL THEN NEW.cost  := m_cost;  END IF;
    IF NEW.price IS NULL THEN NEW.price := m_price; END IF;
  END IF;

  RETURN NEW;
END;
$$;

DROP TRIGGER IF EXISTS trg_store_serialized_enforce_mode ON public.store_serialized_list;
CREATE TRIGGER trg_store_serialized_enforce_mode
BEFORE INSERT OR UPDATE ON public.store_serialized_list
FOR EACH ROW
EXECUTE FUNCTION public.store_serialized_enforce_mode();

-- =============================================
-- Trigger: auto-refresh updated_at (reuses set_updated_at() from file 03)
-- 触发器：自动刷新 updated_at（复用 03_user_list 中的 set_updated_at()）
-- =============================================
DROP TRIGGER IF EXISTS trg_store_serialized_list_updated_at ON public.store_serialized_list;
CREATE TRIGGER trg_store_serialized_list_updated_at
BEFORE UPDATE ON public.store_serialized_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- =============================================
-- Indexes / 索引
-- =============================================

-- Query a store's serialized items filtered by status (e.g., in_stock units for sale)
-- 查询门店的序列号商品，按状态筛选（如查找在库可售的单件）
CREATE INDEX IF NOT EXISTS idx_store_serialized_store_status
  ON public.store_serialized_list (store_id, status);

-- Query all serialized units for a given product (across all stores)
-- 查询某商品的所有序列号单件（跨门店）
CREATE INDEX IF NOT EXISTS idx_store_serialized_unique_id
  ON public.store_serialized_list (unique_id);

-- Quick lookup of active (non-deleted) serialized items in a store
-- 快速查找门店内活跃（未删除）的序列号商品
CREATE INDEX IF NOT EXISTS idx_store_serialized_active
  ON public.store_serialized_list (store_id)
  WHERE deleted_at IS NULL;

-- =============================================
-- =============================================
-- Mother table trigger: mother_inventory_block_mode_change()
-- 母表触发器：mother_inventory_block_mode_change()
-- =============================================
-- =============================================
-- Blocks changes to inventory_mode on mother_inventory_list when active store inventory rows exist.
-- Checks BOTH store_inventory_list (service/tracked/untracked) AND store_serialized_list (serialized).
-- Proper workflow: UI guides the user to soft-delete all store rows first, then change the mode.
-- This trigger is the last line of defense, preventing any entry point (UI/API/script) from bypassing.
-- ─────────────────────────────────────────────
-- 当有活跃的门店库存行时，拦截母表 inventory_mode 变更。
-- 同时检查 store_inventory_list（service/tracked/untracked）和 store_serialized_list（serialized）。
-- 正常流程：UI 引导用户先将所有门店的该商品行软删除（或迁移），再修改母表 mode。
-- 本触发器作为最后一道防线，杜绝任何入口（UI/API/脚本）绕过本约束。
-- =============================================
CREATE OR REPLACE FUNCTION public.mother_inventory_block_mode_change()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  -- Variable legend / 变量说明:
  --   active_inv = count of active rows in store_inventory_list / store_inventory_list 中的活跃行数
  --   active_ser = count of active rows in store_serialized_list / store_serialized_list 中的活跃行数
  active_inv  int;
  active_ser  int;
BEGIN
  -- Only intercept when inventory_mode actually changes
  -- 仅在 inventory_mode 实际发生变化时才拦截
  IF NEW.inventory_mode IS DISTINCT FROM OLD.inventory_mode THEN

    -- Check store_inventory_list for active rows (service / tracked / untracked)
    -- 检查 store_inventory_list 中是否有活跃行（service / tracked / untracked）
    SELECT COUNT(*)
      INTO active_inv
    FROM public.store_inventory_list
    WHERE unique_id = NEW.unique_id
      AND deleted_at IS NULL;

    -- Check store_serialized_list for active rows (serialized)
    -- 检查 store_serialized_list 中是否有活跃行（serialized）
    SELECT COUNT(*)
      INTO active_ser
    FROM public.store_serialized_list
    WHERE unique_id = NEW.unique_id
      AND deleted_at IS NULL;

    IF (active_inv + active_ser) > 0 THEN
      RAISE EXCEPTION
        'Cannot change inventory_mode for unique_id %: '
        '% active store_inventory row(s) + % active store_serialized row(s) exist. '
        'Soft-delete all store rows first, then retry.',
        NEW.unique_id, active_inv, active_ser;
    END IF;
  END IF;

  RETURN NEW;
END;
$$;

DROP TRIGGER IF EXISTS trg_mother_inventory_block_mode_change ON public.mother_inventory_list;
CREATE TRIGGER trg_mother_inventory_block_mode_change
BEFORE UPDATE ON public.mother_inventory_list
FOR EACH ROW
EXECUTE FUNCTION public.mother_inventory_block_mode_change();


# 表10，batch_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 10 · batch_list — Business day batch table
-- 文件 10 · batch_list — 营业批次表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list (store_id FK)
--   03_user_list  (opened_by_user_id / closed_by_user_id FK)
-- Dependents / 被依赖:
--   11_shift_list (store_id, batch_id composite FK)
-- Shared components created here / 本文件创建的共享组件:
--   Function public.assign_batch_id_per_store() — this table only
-- =============================================
-- Each store's business day batch record. Only one open batch per store at a time.
-- No soft delete: batches are permanent historical records; once closed, they are kept forever.
-- ─────────────────────────────────────────────
-- 每个门店的营业批次记录。同一门店同一时间只能有一个 open batch。
-- 不支持软删除：批次是永久历史记录，关闭后永久保留。
-- =============================================

-- =============================================
-- Function: assign_batch_id_per_store()
-- 函数：assign_batch_id_per_store()
-- =============================================
-- Auto-assigns batch_id per store, starting from 1 and incrementing.
-- Uses advisory lock (namespace 1002) to prevent race conditions.
-- ─────────────────────────────────────────────
-- 按门店自动分配 batch_id，从 1 开始递增。
-- 使用咨询锁（命名空间 1002）防止并发冲突。
-- Advisory lock 1002: see allocation table in 06_mother_inventory_list
-- 咨询锁 1002：分配总表见 06_mother_inventory_list
-- =============================================
CREATE OR REPLACE FUNCTION public.assign_batch_id_per_store()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  -- Variable legend / 变量说明:
  --   next_bid = next batch_id to assign for this store / 本门店下要分配的下一个 batch_id
  next_bid int;
BEGIN
  -- Guard: store_id must not be null / 防御性检查：store_id 不能为空
  IF NEW.store_id IS NULL THEN
    RAISE EXCEPTION 'store_id cannot be null';
  END IF;

  -- Advisory lock namespace 1002, keyed by store_id
  -- 咨询锁命名空间 1002，按 store_id 锁定
  PERFORM pg_advisory_xact_lock(1002, hashtext(NEW.store_id));

  SELECT COALESCE(MAX(batch_id), 0) + 1
    INTO next_bid
  FROM public.batch_list
  WHERE store_id = NEW.store_id;

  NEW.batch_id := next_bid;
  RETURN NEW;
END;
$$;

-- =============================================
-- Table creation / 建表
-- =============================================
CREATE TABLE IF NOT EXISTS public.batch_list (

  -- Store ID, FK to store_list
  -- 门店 ID，外键指向 store_list
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Batch number, auto-increments per store starting from 1 (assigned by trigger)
  -- 批次编号，同一门店内从 1 开始自动递增（由触发器分配）
  batch_id integer NOT NULL,

  -- Whether this batch is currently open (true = open, false = closed)
  -- 批次是否处于开启状态（true = 开启，false = 已关闭）
  is_open boolean NOT NULL DEFAULT true,

  -- Timestamp when the batch was opened
  -- 批次开启时间
  opened_at timestamptz NOT NULL DEFAULT now(),

  -- Timestamp when the batch was closed; must be NULL while open
  -- 批次关闭时间；开启状态下应为 NULL
  closed_at timestamptz DEFAULT NULL,

  -- Employee who opened this batch, FK to user_list
  -- 开启批次的员工，外键指向 user_list
  opened_by_user_id integer REFERENCES public.user_list(user_id),

  -- Employee who closed this batch, FK to user_list
  -- 关闭批次的员工，外键指向 user_list
  closed_by_user_id integer REFERENCES public.user_list(user_id),

  -- Optional notes / comments about this batch
  -- 备注（可选）
  note text DEFAULT NULL,

  -- Record creation timestamp
  -- 记录创建时间
  created_at timestamptz NOT NULL DEFAULT now(),

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- Composite PK: (store_id, batch_id) uniquely identifies a batch
  -- 联合主键：(store_id, batch_id) 唯一标识一个批次
  CONSTRAINT batch_list_pkey PRIMARY KEY (store_id, batch_id),

  -- CHECK: batch_id must be a positive integer (>= 1)
  -- 约束：batch_id 应为正整数（>= 1）
  CONSTRAINT chk_batch_list_batch_id_positive CHECK (batch_id > 0),

  -- CHECK: open/close consistency — if open, closed_at must be NULL; if closed, closed_at must be set
  -- 约束：开关状态一致性 — 开启时 closed_at 应为空，关闭时应有值
  CONSTRAINT chk_batch_list_open_close_consistency CHECK (
    (is_open = true  AND closed_at IS NULL)
    OR
    (is_open = false AND closed_at IS NOT NULL)
  )
);

-- =============================================
-- Trigger: auto-assign batch_id on INSERT
-- 触发器：INSERT 时自动分配 batch_id
-- =============================================
DROP TRIGGER IF EXISTS trg_batch_list_assign_id ON public.batch_list;
CREATE TRIGGER trg_batch_list_assign_id
BEFORE INSERT ON public.batch_list
FOR EACH ROW
EXECUTE FUNCTION public.assign_batch_id_per_store();

-- =============================================
-- Partial unique index: only one open batch per store at a time
-- 部分唯一索引：同一门店同一时间只能有一个 open batch
-- =============================================
CREATE UNIQUE INDEX IF NOT EXISTS uq_batch_list_one_open_per_store
  ON public.batch_list (store_id)
  WHERE is_open = true;

-- =============================================
-- Index: look up recent batches for a store (ordered by open time descending)
-- 索引：查询某门店的最近批次（按开启时间倒序）
-- =============================================
CREATE INDEX IF NOT EXISTS idx_batch_list_store_opened_at
  ON public.batch_list (store_id, opened_at DESC);

-- =============================================
-- Trigger: auto-refresh updated_at (reuses set_updated_at() from file 03)
-- 触发器：自动刷新 updated_at（复用 03_user_list 中的 set_updated_at()）
-- =============================================
DROP TRIGGER IF EXISTS trg_batch_list_updated_at ON public.batch_list;
CREATE TRIGGER trg_batch_list_updated_at
BEFORE UPDATE ON public.batch_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();


# 表11，shift_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 11 · shift_list — Employee shift table
-- 文件 11 · shift_list — 班次表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list (store_id FK)
--   03_user_list  (user_id FK)
--   10_batch_list (store_id, batch_id composite FK)
-- Dependents / 被依赖:
--   12_transaction_list (store_id, batch_id, shift_id composite FK)
-- Shared components created here / 本文件创建的共享组件:
--   Function public.assign_shift_id_per_store() — this table only
-- =============================================
-- Each shift belongs to a batch. shift_id is global per store (NOT reset per batch).
-- Only one open shift per device at a time.
-- No soft delete: shifts are permanent historical records.
-- ─────────────────────────────────────────────
-- 每个班次归属于某个 batch。shift_id 在门店内全局递增（不随 batch 重置）。
-- 同一设备同一时间只能有一个 open shift。
-- 不支持软删除：班次是永久历史记录。
-- =============================================

-- =============================================
-- Function: assign_shift_id_per_store()
-- 函数：assign_shift_id_per_store()
-- =============================================
-- Auto-assigns shift_id per store, starting from 1 and incrementing globally (not per batch).
-- Uses advisory lock (namespace 1003) to prevent race conditions.
-- ─────────────────────────────────────────────
-- 按门店自动分配 shift_id，从 1 开始全局递增（不按 batch 重置）。
-- 使用咨询锁（命名空间 1003）防止并发冲突。
-- Advisory lock 1003: see allocation table in 06_mother_inventory_list
-- 咨询锁 1003：分配总表见 06_mother_inventory_list
-- =============================================
CREATE OR REPLACE FUNCTION public.assign_shift_id_per_store()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  -- Variable legend / 变量说明:
  --   next_sid = next shift_id to assign for this store / 本门店下要分配的下一个 shift_id
  next_sid int;
BEGIN
  -- Guard: store_id and batch_id must not be null
  -- 防御性检查：store_id 和 batch_id 不能为空
  IF NEW.store_id IS NULL OR NEW.batch_id IS NULL THEN
    RAISE EXCEPTION 'store_id and batch_id cannot be null';
  END IF;

  -- Advisory lock namespace 1003, keyed by store_id
  -- 咨询锁命名空间 1003，按 store_id 锁定
  PERFORM pg_advisory_xact_lock(1003, hashtext(NEW.store_id));

  SELECT COALESCE(MAX(shift_id), 0) + 1
    INTO next_sid
  FROM public.shift_list
  WHERE store_id = NEW.store_id;

  NEW.shift_id := next_sid;
  RETURN NEW;
END;
$$;

-- =============================================
-- Table creation / 建表
-- =============================================
CREATE TABLE IF NOT EXISTS public.shift_list (

  -- Store ID (part of composite PK and FK to batch_list)
  -- 门店 ID（联合主键的一部分，同时参与 batch_list 外键）
  store_id text NOT NULL,

  -- Batch number this shift belongs to (FK to batch_list via composite key)
  -- 所属批次编号（通过复合键外键指向 batch_list）
  batch_id integer NOT NULL,

  -- Shift number, globally incremented per store (NOT reset per batch), assigned by trigger
  -- 班次编号，门店内全局递增（不随 batch 重置），由触发器分配
  shift_id integer NOT NULL,

  -- Employee on this shift, FK to user_list
  -- 当班员工，外键指向 user_list
  user_id integer REFERENCES public.user_list(user_id),

  -- Device ID used for this shift (e.g., a specific POS terminal)
  -- 本班次使用的设备 ID（如某台 POS 终端）
  device_id text DEFAULT NULL,

  -- Whether this shift is currently open (true = open, false = closed)
  -- 班次是否处于开启状态（true = 开启，false = 已关闭）
  is_open boolean NOT NULL DEFAULT true,

  -- Timestamp when the shift was opened
  -- 班次开启时间
  opened_at timestamptz NOT NULL DEFAULT now(),

  -- Timestamp when the shift was closed; must be NULL while open
  -- 班次关闭时间；开启状态下应为 NULL
  closed_at timestamptz DEFAULT NULL,

  -- Starting cash in the drawer when the shift opened
  -- 开班时的现金底数
  opening_cash numeric(10,2) NOT NULL DEFAULT 0,

  -- Actual cash counted when the shift closed; may be NULL while still open
  -- 关班时的现金实点数；开班状态下允许为 NULL
  closing_cash numeric(10,2) DEFAULT NULL,

  -- Optional notes / comments about this shift
  -- 备注（可选）
  note text DEFAULT NULL,

  -- Record creation timestamp
  -- 记录创建时间
  created_at timestamptz NOT NULL DEFAULT now(),

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- Composite PK: (store_id, batch_id, shift_id)
  -- 三列联合主键
  CONSTRAINT shift_list_pkey PRIMARY KEY (store_id, batch_id, shift_id),

  -- CHECK: shift_id must be a positive integer (>= 1)
  -- 约束：shift_id 应为正整数（>= 1）
  CONSTRAINT chk_shift_list_shift_id_positive CHECK (shift_id > 0),

  -- CHECK: open/close consistency — if open, closed_at must be NULL; if closed, closed_at must be set
  -- 约束：开关状态一致性 — 开启时 closed_at 应为空，关闭时应有值
  CONSTRAINT chk_shift_list_open_close_consistency CHECK (
    (is_open = true  AND closed_at IS NULL)
    OR
    (is_open = false AND closed_at IS NOT NULL)
  ),

  -- Composite FK to batch_list (store_id, batch_id)
  -- 复合外键指向 batch_list (store_id, batch_id)
  CONSTRAINT fk_shift_list_batch_list
    FOREIGN KEY (store_id, batch_id)
    REFERENCES public.batch_list (store_id, batch_id)
);

-- =============================================
-- Trigger: auto-assign shift_id on INSERT
-- 触发器：INSERT 时自动分配 shift_id
-- =============================================
DROP TRIGGER IF EXISTS trg_shift_list_assign_id ON public.shift_list;
CREATE TRIGGER trg_shift_list_assign_id
BEFORE INSERT ON public.shift_list
FOR EACH ROW
EXECUTE FUNCTION public.assign_shift_id_per_store();

-- =============================================
-- Partial unique index: only one open shift per device at a time (within a store)
-- Records with device_id IS NULL are excluded (allows shifts without a device)
-- ─────────────────────────────────────────────
-- 部分唯一索引：同一设备同一时间只能有一个 open shift（门店内）
-- device_id IS NULL 的记录不参与（允许无设备的班次）
-- =============================================
CREATE UNIQUE INDEX IF NOT EXISTS uq_shift_list_one_open_per_device
  ON public.shift_list (store_id, device_id)
  WHERE is_open = true AND device_id IS NOT NULL;

-- =============================================
-- Indexes / 索引
-- =============================================

-- Look up recent shifts for a store (ordered by open time descending)
-- 查询某门店的最近班次（按开启时间倒序）
CREATE INDEX IF NOT EXISTS idx_shift_list_store_opened_at
  ON public.shift_list (store_id, opened_at DESC);

-- Look up shifts within a specific batch
-- 查询某个 batch 下的班次
CREATE INDEX IF NOT EXISTS idx_shift_list_batch_opened_at
  ON public.shift_list (store_id, batch_id, opened_at DESC);

-- =============================================
-- Trigger: auto-refresh updated_at (reuses set_updated_at() from file 03)
-- 触发器：自动刷新 updated_at（复用 03_user_list 中的 set_updated_at()）
-- =============================================
DROP TRIGGER IF EXISTS trg_shift_list_updated_at ON public.shift_list;
CREATE TRIGGER trg_shift_list_updated_at
BEFORE UPDATE ON public.shift_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();


# 表12，transaction_list

```sql
-- Async requirement: YES - offline POS must continue high-frequency essential operations using local snapshot; sync changes to cloud after reconnection.
-- 异步需求：是 - POS 离线时需依赖本地快照继续高频必要操作，网络恢复后将变更同步到云端。
-- =============================================
-- File 12 · transaction_list — Transaction header table
-- 文件 12 · transaction_list — 交易主表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list   (store_id FK)
--   03_user_list    (user_id FK)
--   07_customer_list (customer_id FK)
--   11_shift_list   (store_id, batch_id, shift_id composite FK)
-- Dependents / 被依赖:
--   13_transaction_line_list  (transaction_id FK)
--   16_serialized_event_list  (transaction_id reference, no FK)
-- Shared components created here / 本文件创建的共享组件:
--   ENUM public.transaction_type — this table only
-- =============================================
-- Note: repair_ticket_id FK will be added by ALTER TABLE in file 22_repair_ticket_list
-- 注意：repair_ticket_id 外键将在 22_repair_ticket_list 中通过 ALTER TABLE 添加
-- =============================================

-- =============================================
-- ENUM: transaction_type — Transaction classification
-- 枚举：transaction_type — 交易类型分类
-- =============================================
-- Type is determined by the client based on transaction content and priority rules.
-- The database does NOT validate the type assignment.
-- Priority (high → low): exchange > serialized > repair > sale / refund / payment
-- ─────────────────────────────────────────────
-- 类型由客户端根据交易内容和优先级规则判定，数据库不做校验。
-- 优先级（高→低）：exchange > serialized > repair > sale / refund / payment
--
--   exchange   = Return + re-purchase in the same transaction (any item swap marks it as exchange)
--              = 退货后又换货（只要涉及了任何换购，都标 exchange）
--   serialized = Contains a serialized item sale (priority just below exchange)
--              = 包含序列号商品卖出（优先级仅次于 exchange）
--   repair     = Repair order (only repair parts/services, determined by UI logic)
--              = 修理单（只有修理配件或修理服务，由 UI 逻辑判定）
--   sale       = Standard sales transaction
--              = 最普通的销售交易
--   refund     = Return / refund only (no new purchase)
--              = 仅退货退款
--   payment    = Customer balance operation (top-up, deposit, withdrawal — should be a standalone transaction)
--              = 客户余额变动（充值、存钱、退钱、定金等，应为单独一笔交易）
DO $$
BEGIN
  CREATE TYPE public.transaction_type AS ENUM (
    'exchange', 'serialized', 'repair', 'sale', 'refund', 'payment'
  );
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =============================================
-- Table creation / 建表
-- =============================================
CREATE TABLE IF NOT EXISTS public.transaction_list (

  -- UUID v7 primary key, generated client-side for offline support; no DEFAULT
  -- Printed on receipts as the unique transaction reference
  -- UUID v7 主键，由客户端生成，支持离线；不设 DEFAULT
  -- 小票上会打印本 ID 作为交易唯一凭证
  transaction_id uuid PRIMARY KEY,

  -- Store ID, FK to store_list
  -- 门店 ID，外键指向 store_list
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Customer ID (UUID), FK to customer_list; NULL = walk-in customer
  -- 客户 ID（UUID），外键指向 customer_list；NULL 表示散客
  customer_id uuid DEFAULT NULL REFERENCES public.customer_list(customer_id),

  -- Employee who processed this transaction, FK to user_list
  -- 经手人 ID，外键指向 user_list；每笔交易应有经手人
  user_id integer NOT NULL REFERENCES public.user_list(user_id),

  -- Human-readable document number for receipts
  -- Format: {store_code}{device_no}S-{YYMMDD}-{NNN} (e.g. D2S-260303-003)
  -- NULL allowed for legacy rows / offline temp rows before server normalization
  -- 小票可读单号
  -- 格式：{store_code}{device_no}S-{YYMMDD}-{NNN}（例如 D2S-260303-003）
  -- 兼容历史数据与离线临时记录：允许为 NULL
  display_no text DEFAULT NULL,

  -- Legacy per-store sequential integer (kept for backward compatibility)
  -- 历史整型展示编号（为兼容旧逻辑保留）
  store_transaction_id integer DEFAULT NULL,

  -- Batch ID (filled on sync; may be NULL while offline)
  -- 批次 ID（联网同步时填入，离线时可能为 NULL）
  batch_id integer DEFAULT NULL,

  -- Shift ID (filled on sync; may be NULL while offline)
  -- 班次 ID（联网同步时填入，离线时可能为 NULL）
  shift_id integer DEFAULT NULL,

  -- Related transaction ID: for refund/exchange, points to the original transaction
  -- No FK constraint: the original transaction may not yet be synced (offline scenario)
  -- 关联交易 ID：退货/换货时指向原交易的 transaction_id
  -- 不加外键约束：关联的原交易可能尚未同步到服务端（离线场景）
  related_transaction_id uuid DEFAULT NULL,

  -- Transaction type, determined by client using priority rules (see ENUM above)
  -- 交易类型，由客户端根据优先级规则判定（参见上方枚举）
  type public.transaction_type NOT NULL,

  -- Last edit timestamp; NULL = never edited, non-NULL = time of most recent edit
  -- 编辑时间戳；NULL 表示未编辑过，有值 = 最近一次编辑时间
  edited_at timestamptz DEFAULT NULL,

  -- =============================================
  -- Payment method breakdown (all NOT NULL DEFAULT 0)
  -- 支付方式金额明细（均 NOT NULL DEFAULT 0）
  -- =============================================

  -- Cash payment amount
  -- 现金支付金额
  cash numeric(10,2) NOT NULL DEFAULT 0,

  -- Credit card payment amount
  -- 信用卡支付金额
  credit numeric(10,2) NOT NULL DEFAULT 0,

  -- Debit card payment amount
  -- 借记卡支付金额
  debit numeric(10,2) NOT NULL DEFAULT 0,

  -- Amount paid from customer's store balance
  -- 从客户余额中扣除的金额
  balance numeric(10,2) NOT NULL DEFAULT 0,

  -- =============================================
  -- Summary amounts (all NOT NULL DEFAULT 0)
  -- 汇总金额（均 NOT NULL DEFAULT 0）
  -- =============================================

  -- Transaction total = cash + credit + debit + balance (validated by CHECK below)
  -- 交易总额 = cash + credit + debit + balance（由下方 CHECK 约束校验）
  amount_total numeric(10,2) NOT NULL DEFAULT 0,

  -- Total tax amount
  -- 税额合计
  tax_total numeric(10,2) NOT NULL DEFAULT 0,

  -- Total cost (even for payment-type transactions, cost remains 0, not NULL)
  -- 成本合计（即使是 payment 类型，cost 仍为 0，不为 NULL）
  cost_total numeric(10,2) NOT NULL DEFAULT 0,

  -- Profit = amount_total - tax_total - cost_total (validated by CHECK below)
  -- 利润 = amount_total - tax_total - cost_total（由下方 CHECK 约束校验）
  profit_total numeric(10,2) NOT NULL DEFAULT 0,

  -- =============================================
  -- Notes / 备注
  -- =============================================

  -- Note printed on the customer receipt (customer-visible)
  -- 打印在小票上的备注（客户可见）
  note_on_receipt text DEFAULT NULL,

  -- Internal system note (staff-only, not printed)
  -- 系统内部备注（仅员工可见，不打印）
  note_on_system text DEFAULT NULL,

  -- =============================================
  -- Device / 设备
  -- =============================================

  -- Device ID: generated by the client on first launch and permanently saved
  -- 设备 ID：客户端首次启动时生成并永久保存
  device_id text DEFAULT NULL,

  -- repair_ticket_id will be added by ALTER TABLE in file 22_repair_ticket_list
  -- repair_ticket_id 将在 22_repair_ticket_list 中通过 ALTER TABLE 添加

  -- =============================================
  -- Timestamps / 时间戳
  -- =============================================

  -- Client-side creation timestamp (provided by client for offline scenarios)
  -- 客户端本地创建时间（离线场景下由客户端提供）
  created_at timestamptz NOT NULL,

  -- First sync timestamp to Supabase; NULL = not yet synced
  -- 首次同步到 Supabase 的时间；NULL 表示尚未同步
  synced_at timestamptz DEFAULT NULL,

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL,

  -- Soft delete timestamp; NULL = active, non-NULL = voided
  -- 软删除时间戳；NULL 表示正常，非 NULL 表示已作废
  deleted_at timestamptz DEFAULT NULL,

  -- =============================================
  -- CHECK constraints: validate client-side calculations to prevent dirty data
  -- 兜底校验：确保客户端计算正确，阻止脏数据入库
  -- =============================================

  -- CHECK: amount_total must equal the sum of all payment methods
  -- 约束：交易总额必须等于所有支付方式之和
  CONSTRAINT chk_transaction_list_amount_total
    CHECK (amount_total = cash + credit + debit + balance),

  -- CHECK: profit_total must equal amount_total minus tax and cost
  -- 约束：利润必须等于总额减去税和成本
  CONSTRAINT chk_transaction_list_profit_total
    CHECK (profit_total = amount_total - tax_total - cost_total),

  -- Optional format guard for display_no
  -- 显示单号格式校验（可空）
  CONSTRAINT chk_transaction_list_display_no_format
    CHECK (
      display_no IS NULL
      OR display_no ~ '^[A-Za-z0-9]+S-[0-9]{6}-[0-9]{3}$'
    )
);

-- =============================================
-- Composite FK to shift_list (store_id, batch_id, shift_id)
-- batch_id and shift_id may be NULL when created offline (FK only validates when all columns are non-NULL)
-- ─────────────────────────────────────────────
-- 复合外键指向 shift_list (store_id, batch_id, shift_id)
-- 离线创建时 batch_id 和 shift_id 可能为 NULL（FK 仅在所有列非空时校验）
-- =============================================
DO $$
BEGIN
  ALTER TABLE public.transaction_list
    ADD CONSTRAINT fk_transaction_list_shift_list
    FOREIGN KEY (store_id, batch_id, shift_id)
    REFERENCES public.shift_list (store_id, batch_id, shift_id);
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =============================================
-- Migration safety patch: add display_no for existing databases
-- 迁移兼容补丁：为已存在数据库补充 display_no 列
-- =============================================
ALTER TABLE public.transaction_list
  ADD COLUMN IF NOT EXISTS display_no text;

DO $$
BEGIN
  ALTER TABLE public.transaction_list
    ADD CONSTRAINT chk_transaction_list_display_no_format
    CHECK (
      display_no IS NULL
      OR display_no ~ '^[A-Za-z0-9]+S-[0-9]{6}-[0-9]{3}$'
    );
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =============================================
-- Partial unique index: display_no must be unique within a store (active rows)
-- 部分唯一索引：display_no 在门店内唯一（仅活跃且非空记录）
-- =============================================
CREATE UNIQUE INDEX IF NOT EXISTS uq_transaction_store_display_no
  ON public.transaction_list (store_id, display_no)
  WHERE display_no IS NOT NULL AND deleted_at IS NULL;

-- =============================================
-- Partial unique index: store_transaction_id must be unique within a store (only when assigned)
-- 部分唯一索引：store_transaction_id 在门店内唯一（仅在已分配时生效）
-- =============================================
CREATE UNIQUE INDEX IF NOT EXISTS uq_store_transaction_id
  ON public.transaction_list (store_id, store_transaction_id)
  WHERE store_transaction_id IS NOT NULL;

-- =============================================
-- Trigger: auto-refresh updated_at (reuses set_updated_at() from file 03)
-- 触发器：自动刷新 updated_at（复用 03_user_list 中的 set_updated_at()）
-- =============================================
DROP TRIGGER IF EXISTS trg_transaction_list_updated_at ON public.transaction_list;
CREATE TRIGGER trg_transaction_list_updated_at
BEFORE UPDATE ON public.transaction_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- =============================================
-- Indexes / 索引
-- =============================================

-- Most common query: list transactions for a store (active only, newest first)
-- 最常见查询：列出门店的交易（仅活跃记录，按时间倒序）
CREATE INDEX IF NOT EXISTS idx_transaction_store
  ON public.transaction_list (store_id, created_at DESC)
  WHERE deleted_at IS NULL;

-- Look up transaction history for a specific customer
-- 查询某客户的交易历史
CREATE INDEX IF NOT EXISTS idx_transaction_customer
  ON public.transaction_list (customer_id)
  WHERE deleted_at IS NULL AND customer_id IS NOT NULL;

-- Filter transactions by type within a store
-- 按交易类型筛选（门店内）
CREATE INDEX IF NOT EXISTS idx_transaction_type
  ON public.transaction_list (store_id, type)
  WHERE deleted_at IS NULL;

-- Look up transactions by employee
-- 按经手人查交易
CREATE INDEX IF NOT EXISTS idx_transaction_user
  ON public.transaction_list (user_id)
  WHERE deleted_at IS NULL;

-- Look up related transactions (for refund/exchange tracing)
-- 查询关联交易（退货/换货溯源）
CREATE INDEX IF NOT EXISTS idx_transaction_related
  ON public.transaction_list (related_transaction_id)
  WHERE related_transaction_id IS NOT NULL;


# 表13，transaction_line_list

```sql
-- Async requirement: YES - offline POS must continue high-frequency essential operations using local snapshot; sync changes to cloud after reconnection.
-- 异步需求：是 - POS 离线时需依赖本地快照继续高频必要操作，网络恢复后将变更同步到云端。
-- =============================================
-- File 13 · transaction_line_list — Transaction line item table
-- 文件 13 · transaction_line_list — 交易明细行表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list            (store_id FK)
--   06_mother_inventory_list (unique_id FK + inventory_mode ENUM)
--   09_store_serialized_list (unit_id FK)
--   12_transaction_list      (transaction_id FK)
-- Dependents / 被依赖:
--   (none) （无）
-- =============================================
-- Each transaction (transaction_list) contains one or more line items.
-- Supports offline creation: PK is a UUID v7 generated client-side.
-- All monetary values are calculated by the client; the database validates via CHECK constraints.
-- ─────────────────────────────────────────────
-- 每笔交易（transaction_list）包含一条或多条明细行。
-- 支持离线创建：主键由客户端生成 UUID v7。
-- 所有金额由客户端计算，数据库通过 CHECK 约束兜底校验。
-- =============================================

CREATE TABLE IF NOT EXISTS public.transaction_line_list (

  -- UUID v7 primary key, generated client-side for offline support
  -- UUID v7 主键，由客户端生成，支持离线
  line_id uuid PRIMARY KEY,

  -- Parent transaction, FK to transaction_list
  -- 所属交易，外键指向 transaction_list
  transaction_id uuid NOT NULL REFERENCES public.transaction_list(transaction_id),

  -- Store ID (denormalized to avoid frequent JOINs with transaction_list for queries)
  -- 门店 ID（冗余字段，避免高频查询时 JOIN transaction_list）
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Product reference, FK to mother_inventory_list (required — every line must map to a catalog item)
  -- 母表商品引用，外键指向 mother_inventory_list（必填 — 每条明细必须对应一个目录商品）
  unique_id integer NOT NULL REFERENCES public.mother_inventory_list(unique_id),

  -- Serialized unit reference: required for serialized items, must be NULL for non-serialized
  -- Consistency validated by trigger (see Trigger A below)
  -- 序列号单件引用：serialized 商品必填，非 serialized 商品应为 NULL
  -- 一致性由触发器校验（见下方触发器 A）
  unit_id integer DEFAULT NULL REFERENCES public.store_serialized_list(unit_id),

  -- Product name snapshot: copied from catalog by the client at transaction time
  -- The client may allow manual edits; DB does NOT auto-inherit (offline can't access catalog)
  -- 商品名称快照：交易时由客户端从目录复制
  -- 客户端 UI 层面允许修改；数据库不做自动继承触发器（离线场景下无法访问母表）
  item_name text NOT NULL,

  -- Quantity: can be 0 (pure service item) or negative (return/refund line)
  -- 数量：可以为 0（纯服务项）或负数（退货行）
  qty integer NOT NULL,

  -- Cost per unit: read by client from store_inventory_list or store_serialized_list
  -- 单位成本：由客户端从 store_inventory_list 或 store_serialized_list 读取
  cost_per_unit numeric(10,2) NOT NULL DEFAULT 0,

  -- Selling price per unit: read by client from store_inventory_list or store_serialized_list
  -- 单位售价：由客户端从 store_inventory_list 或 store_serialized_list 读取
  price_per_unit numeric(10,2) NOT NULL DEFAULT 0,

  -- Discount amount for this line (positive value = money off)
  -- 该行折扣金额（正数表示优惠金额）
  line_discount numeric(10,2) NOT NULL DEFAULT 0,

  -- Tax amount for this line
  -- 该行税额
  line_tax numeric(10,2) NOT NULL DEFAULT 0,

  -- Pre-tax line total = qty × price_per_unit − line_discount (validated by CHECK)
  -- 税前行总额 = qty × price_per_unit − line_discount（由 CHECK 约束校验）
  line_total_before_tax numeric(10,2) NOT NULL DEFAULT 0,

  -- Line total including tax = line_total_before_tax + line_tax (validated by CHECK)
  -- 含税行总额 = line_total_before_tax + line_tax（由 CHECK 约束校验）
  line_total numeric(10,2) NOT NULL DEFAULT 0,

  -- Line profit = line_total_before_tax − qty × cost_per_unit (validated by CHECK)
  -- 行利润 = line_total_before_tax − qty × cost_per_unit（由 CHECK 约束校验）
  line_profit numeric(10,2) NOT NULL DEFAULT 0,

  -- Last edit timestamp; NULL = never edited, non-NULL = time of most recent edit
  -- 编辑时间戳；NULL 表示未被编辑过，有值 = 最近一次编辑时间
  edited_at timestamptz DEFAULT NULL,

  -- Soft delete timestamp; NULL = active, non-NULL = voided
  -- 软删除时间戳；NULL 表示正常，非 NULL 表示已作废
  deleted_at timestamptz DEFAULT NULL,

  -- =============================================
  -- CHECK constraints: validate client-side calculations to prevent dirty data
  -- 兜底校验：确保客户端计算正确，阻止脏数据入库
  -- =============================================

  -- CHECK: line_total_before_tax = qty × price_per_unit − line_discount
  -- 约束：税前行总额 = 数量 × 单价 − 折扣
  CONSTRAINT chk_transaction_line_list_total_before_tax
    CHECK (line_total_before_tax = qty * price_per_unit - line_discount),

  -- CHECK: line_total = line_total_before_tax + line_tax
  -- 约束：含税行总额 = 税前行总额 + 税额
  CONSTRAINT chk_transaction_line_list_total
    CHECK (line_total = line_total_before_tax + line_tax),

  -- CHECK: line_profit = line_total_before_tax − qty × cost_per_unit
  -- 约束：行利润 = 税前行总额 − 数量 × 单位成本
  CONSTRAINT chk_transaction_line_list_profit
    CHECK (line_profit = line_total_before_tax - qty * cost_per_unit)
);

-- =============================================
-- Trigger A: validate unit_id consistency with inventory_mode
-- 触发器 A：校验 unit_id 与 inventory_mode 的一致性
-- =============================================
-- Serialized items must provide unit_id; non-serialized items must have unit_id = NULL.
-- This trigger fires on sync to Supabase; it does NOT apply during offline operation.
-- ─────────────────────────────────────────────
-- serialized 商品应提供 unit_id，非 serialized 商品 unit_id 应为 NULL。
-- 本触发器在数据同步到 Supabase 时执行，离线期间不生效。
-- =============================================
CREATE OR REPLACE FUNCTION public.transaction_line_enforce_unit_id()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  -- Variable legend / 变量说明:
  --   v_mode = inventory_mode from mother table / 母表的库存跟踪模式
  v_mode public.inventory_mode;
BEGIN
  SELECT inventory_mode
    INTO v_mode
  FROM public.mother_inventory_list
  WHERE unique_id = NEW.unique_id;

  IF NOT FOUND THEN
    RAISE EXCEPTION 'mother_inventory_list.unique_id % not found', NEW.unique_id;
  END IF;

  -- Serialized items must reference a specific unit
  -- serialized 商品应关联具体单件
  IF v_mode = 'serialized' AND NEW.unit_id IS NULL THEN
    RAISE EXCEPTION 'serialized item (unique_id=%) requires unit_id', NEW.unique_id;
  END IF;

  -- Non-serialized items must NOT have a unit_id
  -- 非 serialized 商品不应该有 unit_id
  IF v_mode IS DISTINCT FROM 'serialized' AND NEW.unit_id IS NOT NULL THEN
    RAISE EXCEPTION 'non-serialized item (unique_id=%) must not have unit_id', NEW.unique_id;
  END IF;

  RETURN NEW;
END;
$$;

DROP TRIGGER IF EXISTS trg_transaction_line_enforce_unit_id ON public.transaction_line_list;
CREATE TRIGGER trg_transaction_line_enforce_unit_id
BEFORE INSERT OR UPDATE ON public.transaction_line_list
FOR EACH ROW
EXECUTE FUNCTION public.transaction_line_enforce_unit_id();

-- =============================================
-- Indexes / 索引
-- =============================================

-- Most frequent query: list all line items for a transaction (active only)
-- 最高频查询：列出某笔交易的所有明细行（仅活跃记录）
CREATE INDEX IF NOT EXISTS idx_transaction_line_transaction
  ON public.transaction_line_list (transaction_id)
  WHERE deleted_at IS NULL;

-- List all line items for a store (report scenarios)
-- 列出门店的所有明细行（报表场景）
CREATE INDEX IF NOT EXISTS idx_transaction_line_store
  ON public.transaction_line_list (store_id)
  WHERE deleted_at IS NULL;

-- Look up sales history for a specific product
-- 查询某商品的销售记录
CREATE INDEX IF NOT EXISTS idx_transaction_line_unique_id
  ON public.transaction_line_list (unique_id)
  WHERE deleted_at IS NULL;

-- Look up transaction history for a specific serialized unit
-- 查询某序列号单件的交易记录
CREATE INDEX IF NOT EXISTS idx_transaction_line_unit_id
  ON public.transaction_line_list (unit_id)
  WHERE unit_id IS NOT NULL AND deleted_at IS NULL;


# 表14，store_inventory_adjustment_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 14 · store_inventory_adjustment_list — Inventory adjustment header table
-- 文件 14 · store_inventory_adjustment_list — 库存盘点/调整主表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list (store_id FK)
--   03_user_list  (user_id FK)
-- Dependents / 被依赖:
--   15_store_inventory_adjustment_line_list (adjustment_id FK)
-- =============================================
-- Records the header info for each inventory adjustment operation:
-- who did it, which store, when, and the total cost impact.
-- Online-only (no synced_at field — adjustments cannot be performed offline).
-- By design, only INSERT is intended (no UPDATE or DELETE), but the DB does not
-- enforce immutability here to allow future flexibility.
-- ─────────────────────────────────────────────
-- 记录每次库存盘点/调整操作的基本信息：操作人、门店、时间、总成本变动。
-- 仅在联网时可操作，不支持离线（无 synced_at 字段）。
-- 设计上只允许 INSERT，不应 UPDATE 或 DELETE
-- （但数据库层面不做强制拦截，留后余地）。
-- =============================================

CREATE TABLE IF NOT EXISTS public.store_inventory_adjustment_list (

  -- Auto-increment primary key, starting from 0
  -- 全局自增主键，从 0 开始
  adjustment_id integer
    GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0)
    PRIMARY KEY,

  -- Store where this adjustment was performed, FK to store_list
  -- 执行盘点的门店，外键指向 store_list
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Employee who performed this adjustment, FK to user_list
  -- 执行盘点的员工，外键指向 user_list
  user_id integer NOT NULL REFERENCES public.user_list(user_id),

  -- Total cost impact of this adjustment (sum of all line item cost deltas)
  -- Calculated by the client and passed in; the DB does NOT auto-aggregate
  -- 本次盘点造成的总成本变动（所有明细行的成本差额汇总）
  -- 由客户端计算后传入，数据库不做自动汇总
  cost_delta numeric(10,2) NOT NULL DEFAULT 0,

  -- Reason / description for this adjustment (e.g., "Monthly stocktake", "Damage write-off")
  -- 盘点备注/原因说明（如"月度盘点"、"损坏报废"）
  description text DEFAULT NULL,

  -- Creation timestamp (server time, since this is an online-only operation)
  -- 创建时间（服务端时间，因为此操作仅在联网时进行）
  created_at timestamptz NOT NULL DEFAULT now()
);

-- =============================================
-- Indexes / 索引
-- =============================================

-- Look up adjustment history for a store (newest first)
-- 查询门店的盘点记录（按时间倒序）
CREATE INDEX IF NOT EXISTS idx_adjustment_list_store
  ON public.store_inventory_adjustment_list (store_id, created_at DESC);

-- Look up which adjustments a specific employee performed
-- 查询某员工执行的盘点操作
CREATE INDEX IF NOT EXISTS idx_adjustment_list_user
  ON public.store_inventory_adjustment_list (user_id);


# 表15，store_inventory_adjustment_line_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 15 · store_inventory_adjustment_line_list — Inventory adjustment line item table
-- 文件 15 · store_inventory_adjustment_line_list — 库存盘点/调整明细行表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list            (store_id, via composite FK)
--   06_mother_inventory_list (unique_id, via composite FK + inventory_mode ENUM)
--   08_store_inventory_list  (store_id, unique_id composite FK + stock_bucket ENUM)
--   14_store_inventory_adjustment_list (adjustment_id FK)
-- Dependents / 被依赖:
--   (none) （无）
-- =============================================
-- Each adjustment (header) contains one or more line items, each recording the
-- inventory change for a single product.
-- Only for non-serialized products (service/tracked/untracked; service is rejected by trigger).
-- Serialized items have their own lifecycle management via serialized_event_list.
-- ─────────────────────────────────────────────
-- 每次盘点（主表）包含一条或多条明细行，记录每个商品的库存调整。
-- 仅适用于非 serialized 商品（service/tracked/untracked；service 会被触发器拦截）。
-- serialized 商品有独立的生命周期管理（serialized_event_list）。
-- =============================================

CREATE TABLE IF NOT EXISTS public.store_inventory_adjustment_line_list (

  -- Auto-increment primary key, starting from 0
  -- 全局自增主键，从 0 开始
  line_id integer
    GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0)
    PRIMARY KEY,

  -- Store ID (matches header; also part of composite FK to store_inventory_list)
  -- 门店 ID（与主表一致；同时参与 store_inventory_list 复合外键）
  store_id text NOT NULL,

  -- Parent adjustment, FK to store_inventory_adjustment_list
  -- 所属盘点单，外键指向 store_inventory_adjustment_list
  adjustment_id integer NOT NULL REFERENCES public.store_inventory_adjustment_list(adjustment_id),

  -- Product reference (part of composite FK to store_inventory_list)
  -- 商品引用（与 store_id 一起组成 store_inventory_list 复合外键）
  unique_id integer NOT NULL,

  -- Quantity change for tracked items: new_qty − old_qty
  -- May be 0 (confirming count is correct), may be negative
  -- Must be NULL for untracked / service items
  -- tracked 商品的数量变动：新数量 − 旧数量
  -- 允许为 0（表示盘点确认数量无误）、允许为负
  -- untracked / service 商品应为 NULL
  qty_delta integer DEFAULT NULL,

  -- Post-adjustment quantity snapshot for tracked items
  -- Recorded by the client at adjustment time for audit purposes
  -- tracked 商品调整后的库存数量快照
  -- 由客户端在调整时记录，方便审计查询
  qty_after integer DEFAULT NULL,

  -- Post-adjustment stock level for untracked items (the new fuzzy level)
  -- untracked 商品调整后的模糊库存档位（新档位值）
  stock_bucket_after public.stock_bucket DEFAULT NULL,

  -- Cost per unit snapshot at adjustment time (read from store_inventory_list.cost)
  -- 调整时的单位成本快照（从 store_inventory_list.cost 读取）
  cost_per_unit numeric(10,2) DEFAULT NULL,

  -- Composite FK to store_inventory_list (store_id, unique_id)
  -- 复合外键指向 store_inventory_list (store_id, unique_id)
  CONSTRAINT fk_adjustment_line_list_store_inventory_list
    FOREIGN KEY (store_id, unique_id)
    REFERENCES public.store_inventory_list (store_id, unique_id),

  -- Mutual exclusion: qty_delta and stock_bucket_after cannot both be non-NULL
  -- 互斥约束：qty_delta 和 stock_bucket_after 不能同时有值
  CONSTRAINT chk_adjustment_line_list_qty_bucket_mutex
    CHECK (NOT (qty_delta IS NOT NULL AND stock_bucket_after IS NOT NULL))
);

-- =============================================
-- Trigger: validate inventory_mode consistency
-- 触发器：校验 inventory_mode 一致性
-- =============================================
-- Rules:
--   service    → reject (no inventory to adjust)
--   serialized → reject (use serialized_event_list instead)
--   tracked    → qty_delta required, stock_bucket fields must be NULL
--   untracked  → stock_bucket_after required, qty fields must be NULL
-- 规则：
--   service    → 拒绝（不跟踪库存，盘点无意义）
--   serialized → 拒绝（应使用 serialized_event_list）
--   tracked    → 必须有 qty_delta，stock_bucket 字段必须为 NULL
--   untracked  → 必须有 stock_bucket_after，qty 字段必须为 NULL
-- =============================================
CREATE OR REPLACE FUNCTION public.adjustment_line_enforce_mode()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  -- Variable legend / 变量说明:
  --   v_mode = inventory_mode from mother table / 母表的库存跟踪模式
  v_mode public.inventory_mode;
BEGIN
  SELECT inventory_mode
    INTO v_mode
  FROM public.mother_inventory_list
  WHERE unique_id = NEW.unique_id;

  IF NOT FOUND THEN
    RAISE EXCEPTION 'mother_inventory_list.unique_id % not found', NEW.unique_id;
  END IF;

  -- Service items cannot be adjusted (no inventory to track)
  -- service 商品不可盘点（不跟踪库存）
  IF v_mode = 'service' THEN
    RAISE EXCEPTION 'service item (unique_id=%) cannot be adjusted', NEW.unique_id;
  END IF;

  -- Serialized items cannot be adjusted here (use serialized_event_list)
  -- serialized 商品不可在本表盘点（应使用 serialized_event_list）
  IF v_mode = 'serialized' THEN
    RAISE EXCEPTION 'serialized item (unique_id=%) cannot be adjusted here', NEW.unique_id;
  END IF;

  -- Tracked items: must have qty_delta, must NOT have stock_bucket fields
  -- tracked 商品：应有 qty_delta，不应有 bucket 字段
  IF v_mode = 'tracked' THEN
    IF NEW.qty_delta IS NULL THEN
      RAISE EXCEPTION 'tracked item (unique_id=%) requires qty_delta', NEW.unique_id;
    END IF;
    IF NEW.stock_bucket_after IS NOT NULL THEN
      RAISE EXCEPTION 'tracked item (unique_id=%) must not have stock_bucket fields', NEW.unique_id;
    END IF;
  END IF;

  -- Untracked items: must have stock_bucket_after, must NOT have qty fields
  -- untracked 商品：应有 stock_bucket_after，不应有 qty 字段
  IF v_mode = 'untracked' THEN
    IF NEW.stock_bucket_after IS NULL THEN
      RAISE EXCEPTION 'untracked item (unique_id=%) requires stock_bucket_after', NEW.unique_id;
    END IF;
    IF NEW.qty_delta IS NOT NULL OR NEW.qty_after IS NOT NULL THEN
      RAISE EXCEPTION 'untracked item (unique_id=%) must not have qty fields', NEW.unique_id;
    END IF;
  END IF;

  RETURN NEW;
END;
$$;

DROP TRIGGER IF EXISTS trg_adjustment_line_enforce_mode ON public.store_inventory_adjustment_line_list;
CREATE TRIGGER trg_adjustment_line_enforce_mode
BEFORE INSERT OR UPDATE ON public.store_inventory_adjustment_line_list
FOR EACH ROW
EXECUTE FUNCTION public.adjustment_line_enforce_mode();

-- =============================================
-- Indexes / 索引
-- =============================================

-- Look up all line items for a given adjustment
-- 查询某次盘点的所有明细行
CREATE INDEX IF NOT EXISTS idx_adjustment_line_adjustment
  ON public.store_inventory_adjustment_line_list (adjustment_id);

-- Look up adjustment history for a specific product
-- 查询某商品的盘点历史
CREATE INDEX IF NOT EXISTS idx_adjustment_line_unique_id
  ON public.store_inventory_adjustment_line_list (unique_id);

-- Look up all adjustment lines for a store
-- 查询门店的所有盘点明细
CREATE INDEX IF NOT EXISTS idx_adjustment_line_store
  ON public.store_inventory_adjustment_line_list (store_id);


# 表16，serialized_event_list

```sql
-- Async requirement: YES - offline POS must continue high-frequency essential operations using local snapshot; sync changes to cloud after reconnection.
-- 异步需求：是 - POS 离线时需依赖本地快照继续高频必要操作，网络恢复后将变更同步到云端。
-- =============================================
-- File 16 · serialized_event_list — Serialized item lifecycle event log
-- 文件 16 · serialized_event_list — 序列号商品生命周期事件表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list            (store_id FK)
--   03_user_list             (user_id FK)
--   09_store_serialized_list (unit_id FK + serialized_status ENUM)
-- Dependents / 被依赖:
--   (none) （无）
-- Shared components created here / 本文件创建的共享组件:
--   ENUM public.serialized_event_type — this table only
--   Uses ENUM public.serialized_status created in file 09
-- =============================================
-- Immutable audit log recording every lifecycle event for serialized items (e.g., phones).
-- Supports offline creation: PK is a UUID v7 generated client-side.
-- By design, only INSERT is intended (no UPDATE or DELETE), but the DB does not
-- enforce immutability here to allow future flexibility.
-- ─────────────────────────────────────────────
-- 不可变审计日志，记录每一件序列号商品（如手机）的完整生命线（状态变化历史）。
-- 支持离线创建：主键由客户端生成 UUID v7。
-- 设计上只允许 INSERT，不应 UPDATE 或 DELETE
-- （但数据库层面不做强制拦截，留后余地）。
-- =============================================

-- =============================================
-- Note: serialized_status is defined in file 09 and already includes lost/wasted.
-- 说明：serialized_status 在 09 文件中定义，且已包含 lost/wasted。
-- =============================================

-- =============================================
-- ENUM: serialized_event_type — Event types for serialized item lifecycle
-- 枚举：serialized_event_type — 序列号商品生命周期事件类型
-- =============================================
-- Each event_type corresponds to a status transition in serialized_status:
-- 每种 event_type 对应 serialized_status 的一次状态迁移：
--
--   purchase              : → in_stock          New unit received into inventory
--                         = 新货入库，在 store_serialized_list 创建新行
--   sell                  : in_stock → sold      Sold to a customer via transaction
--                         = 通过交易卖出给客户
--   return                : sold → in_stock      Customer returned the unit
--                         = 客户退货回库
--   mark_as_sold          : in_stock → sold      Manually marked as sold (inventory correction, not via transaction)
--                         = 手动标记为已售（库存修正，非交易触发）
--   mark_as_lost          : in_stock → lost      Marked as lost
--                         = 标记为丢失
--   mark_as_wasted        : in_stock → wasted    Marked as damaged/wasted
--                         = 标记为损坏/报废
--   store_transferred     : in_stock → in_transit  Sent to another store (transfer initiated)
--                         = 调拨发出到其他门店（调拨发起）
--   transferred_accepted  : in_transit → in_stock  Received from another store (store_id changes)
--                         = 从其他门店接收（store_id 变更）
--   repair_out            : in_stock → repair    Sent out for external repair
--                         = 送外部维修
--   repair_in             : repair → in_stock    Returned from external repair
--                         = 外部维修返回
--   delete                : in_stock → void      Logically deleted (voided)
--                         = 逻辑删除（作废）
--   revive                : void → in_stock      Restored from voided state
--                         = 从作废状态恢复
--   serial_edit           : (no status change)   Serial number was edited
--                         = （无状态变化）序列号被修改
DO $$
BEGIN
  CREATE TYPE public.serialized_event_type AS ENUM (
    'purchase',
    'sell',
    'return',
    'mark_as_sold',
    'mark_as_lost',
    'mark_as_wasted',
    'store_transferred',
    'transferred_accepted',
    'repair_out',
    'repair_in',
    'delete',
    'revive',
    'serial_edit'
  );
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =============================================
-- Table creation / 建表
-- =============================================
CREATE TABLE IF NOT EXISTS public.serialized_event_list (

  -- UUID v7 primary key, generated client-side for offline support
  -- UUID v7 主键，由客户端生成，支持离线
  event_id uuid PRIMARY KEY,

  -- Event type (see ENUM definition above)
  -- 事件类型（参见上方枚举定义）
  event_type public.serialized_event_type NOT NULL,

  -- Store where this event occurred, FK to store_list
  -- 事件发生的门店，外键指向 store_list
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Serialized unit this event applies to, FK to store_serialized_list
  -- 本事件涉及的序列号单件，外键指向 store_serialized_list
  unit_id integer NOT NULL REFERENCES public.store_serialized_list(unit_id),

  -- Employee who performed this action, FK to user_list
  -- 执行本操作的员工，外键指向 user_list
  user_id integer NOT NULL REFERENCES public.user_list(user_id),

  -- Related transaction ID: populated for sell/return events, NULL for others
  -- No FK constraint: the related transaction may not yet be synced (offline scenario)
  -- 关联交易 ID：sell/return 事件时有值，其他事件为 NULL
  -- 不加外键约束：关联交易可能尚未同步到服务端（离线场景）
  transaction_id uuid DEFAULT NULL,

  -- Related transaction line ID: populated for sell/return events, NULL for others
  -- No FK constraint: same reason as above
  -- 关联交易明细行 ID：sell/return 事件时有值，其他事件为 NULL
  -- 不加外键约束：同上
  line_id uuid DEFAULT NULL,

  -- Serial number snapshot at event time (provided by client)
  -- For serial_edit events, this is the NEW serial after the edit
  -- 事件发生时的序列号快照（由客户端填写）
  -- 对于 serial_edit 事件，本值为修改后的新 serial
  serial text NOT NULL,

  -- Optional event note
  -- For serial_edit + swap operations, the client records the swapped serial pairs here
  -- 事件备注（可选）
  -- serial_edit + 互换操作时，客户端在此记录互换的两组序列号
  note text DEFAULT NULL,

  -- Client-side creation timestamp (provided by client for offline scenarios)
  -- 客户端本地创建时间（离线场景下由客户端提供）
  created_at timestamptz NOT NULL
);

-- =============================================
-- Indexes / 索引
-- =============================================

-- Look up the complete event timeline for a specific unit (most frequent query)
-- 查询某单件的完整事件时间线（最高频查询）
CREATE INDEX IF NOT EXISTS idx_serialized_event_unit
  ON public.serialized_event_list (unit_id, created_at DESC);

-- Look up all events in a store (newest first)
-- 查询门店的所有事件（按时间倒序）
CREATE INDEX IF NOT EXISTS idx_serialized_event_store
  ON public.serialized_event_list (store_id, created_at DESC);

-- Look up events related to a specific transaction
-- 查询某笔交易关联的事件
CREATE INDEX IF NOT EXISTS idx_serialized_event_transaction
  ON public.serialized_event_list (transaction_id)
  WHERE transaction_id IS NOT NULL;

-- Filter events by type (e.g., find all sell events, all transfer events)
-- 按事件类型筛选（如查找所有卖出事件、所有调拨事件）
CREATE INDEX IF NOT EXISTS idx_serialized_event_type
  ON public.serialized_event_list (event_type);

-- Look up events by the employee who performed them
-- 按操作员工查询事件记录
CREATE INDEX IF NOT EXISTS idx_serialized_event_user
  ON public.serialized_event_list (user_id);


# 表17，store_transfer_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 17 · store_transfer_list — Inter-store transfer header table
-- 文件 17 · store_transfer_list — 门店间调拨主表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list (store_id / target_store_id FK)
--   03_user_list  (created_by_user_id / confirmed_by_user_id FK)
-- Dependents / 被依赖:
--   18_store_transfer_line_list (store_transfer_id FK)
-- =============================================
-- Records inter-store inventory transfers. Supports tracked / untracked / serialized items.
-- Online-only (no synced_at field — transfers cannot be performed offline).
-- Workflow: source store creates the transfer → target store confirms receipt.
-- By design, only INSERT and confirmation UPDATEs are intended (no DELETE).
-- ─────────────────────────────────────────────
-- 记录门店间的库存调拨单。支持 tracked / untracked / serialized 商品。
-- 仅在联网时可操作，不支持离线。
-- 流程：发货门店创建调拨单 → 收货门店确认收货。
-- 设计上只允许 INSERT 和确认收货时的 UPDATE，不应删除。
-- =============================================

CREATE TABLE IF NOT EXISTS public.store_transfer_list (

  -- Auto-increment primary key, starting from 0
  -- 全局自增主键，从 0 开始
  store_transfer_id integer
    GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0)
    PRIMARY KEY,

  -- Source store (sender), FK to store_list
  -- 调出门店（发货方），外键指向 store_list
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Target store (receiver), FK to store_list
  -- 调入门店（收货方），外键指向 store_list
  target_store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Employee who created the transfer, FK to user_list
  -- 创建调拨单的员工，外键指向 user_list
  created_by_user_id integer NOT NULL REFERENCES public.user_list(user_id),

  -- Employee who confirmed receipt; NULL = not yet confirmed (still in transit)
  -- 确认收货的员工；NULL 表示尚未确认（仍在途中）
  confirmed_by_user_id integer DEFAULT NULL REFERENCES public.user_list(user_id),

  -- Timestamp when receipt was confirmed; NULL = not yet confirmed
  -- confirmed_at and confirmed_by_user_id must both be NULL or both be set (see CHECK below)
  -- 确认收货的时间；NULL 表示尚未确认
  -- confirmed_at 和 confirmed_by_user_id 应同时为空或同时有值（见下方 CHECK）
  confirmed_at timestamptz DEFAULT NULL,

  -- Total cost of all items in this transfer (calculated by client)
  -- 调拨总成本（客户端计算后传入）
  cost_total numeric(10,2) NOT NULL DEFAULT 0,

  -- Total retail value of all items in this transfer (calculated by client)
  -- 调拨总零售价值（客户端计算后传入）
  price_total numeric(10,2) NOT NULL DEFAULT 0,

  -- Record creation timestamp
  -- 记录创建时间
  created_at timestamptz NOT NULL DEFAULT now(),

  -- Last update timestamp (auto-refreshed by set_updated_at() trigger)
  -- 最后更新时间（由 set_updated_at() 触发器自动刷新）
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- CHECK: cannot transfer to yourself (source and target must be different stores)
  -- 约束：不能自己调给自己（发货方和收货方必须是不同门店）
  CONSTRAINT chk_store_transfer_list_not_self
    CHECK (store_id != target_store_id),

  -- CHECK: confirmation consistency — confirmed_by_user_id and confirmed_at must both be NULL or both be set
  -- 约束：确认状态一致性 — confirmed_by_user_id 和 confirmed_at 应同时为空或同时有值
  CONSTRAINT chk_store_transfer_list_confirm_consistency
    CHECK (
      (confirmed_by_user_id IS NULL AND confirmed_at IS NULL)
      OR
      (confirmed_by_user_id IS NOT NULL AND confirmed_at IS NOT NULL)
    )
);

-- =============================================
-- Indexes / 索引
-- =============================================

-- Look up transfers sent from a store (newest first)
-- 查询门店发出的调拨单（按时间倒序）
CREATE INDEX IF NOT EXISTS idx_transfer_store
  ON public.store_transfer_list (store_id, created_at DESC);

-- Look up transfers received by a store (newest first)
-- 查询门店收到的调拨单（按时间倒序）
CREATE INDEX IF NOT EXISTS idx_transfer_target_store
  ON public.store_transfer_list (target_store_id, created_at DESC);

-- Look up pending (unconfirmed) transfers waiting for a store to confirm
-- 查询等待某门店确认的调拨单（未确认 = 仍在途中）
CREATE INDEX IF NOT EXISTS idx_transfer_pending
  ON public.store_transfer_list (target_store_id)
  WHERE confirmed_at IS NULL;

-- =============================================
-- Trigger: auto-refresh updated_at (reuses set_updated_at() from file 03)
-- 触发器：自动刷新 updated_at（复用 03_user_list 中的 set_updated_at()）
-- =============================================
DROP TRIGGER IF EXISTS trg_store_transfer_list_updated_at ON public.store_transfer_list;
CREATE TRIGGER trg_store_transfer_list_updated_at
BEFORE UPDATE ON public.store_transfer_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();


# 表18，store_transfer_line_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 18 · store_transfer_line_list — Inter-store transfer line item table
-- 文件 18 · store_transfer_line_list — 调拨明细行表
-- =============================================
-- Dependencies / 依赖:
--   06_mother_inventory_list (unique_id FK + inventory_mode ENUM)
--   09_store_serialized_list (unit_id FK)
--   17_store_transfer_list   (store_transfer_id FK)
-- Dependents / 被依赖:
--   (none) （无）
-- =============================================
-- Each transfer (header) contains one or more line items.
-- Supports tracked (qty > 1 allowed), untracked (qty typically 1), and serialized (qty must be 1 with unit_id + serial).
-- ─────────────────────────────────────────────
-- 每次调拨（主表）包含一条或多条明细行。
-- 支持 tracked（qty 可大于 1）、untracked（qty 通常为 1）和 serialized（qty 必须为 1，需有 unit_id + serial）。
-- =============================================

CREATE TABLE IF NOT EXISTS public.store_transfer_line_list (

  -- Auto-increment primary key, starting from 0
  -- 全局自增主键，从 0 开始
  line_id integer
    GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0)
    PRIMARY KEY,

  -- Parent transfer, FK to store_transfer_list
  -- 所属调拨单，外键指向 store_transfer_list
  store_transfer_id integer NOT NULL REFERENCES public.store_transfer_list(store_transfer_id),

  -- Product reference, FK to mother_inventory_list
  -- 商品引用，外键指向 mother_inventory_list
  unique_id integer NOT NULL REFERENCES public.mother_inventory_list(unique_id),

  -- Serialized unit reference: required for serialized items, must be NULL for non-serialized
  -- Consistency validated by trigger (see below)
  -- 序列号单件引用：serialized 商品必填，非 serialized 商品应为 NULL
  -- 一致性由触发器校验（见下方）
  unit_id integer DEFAULT NULL REFERENCES public.store_serialized_list(unit_id),

  -- Product name snapshot (denormalized, copied by client at transfer time)
  -- 商品名称快照（冗余字段，由客户端在调拨时复制）
  item_name text NOT NULL,

  -- Cost per unit snapshot at transfer time
  -- 调拨时的单位成本快照
  cost_per_unit numeric(10,2) NOT NULL DEFAULT 0,

  -- Retail price per unit snapshot at transfer time
  -- 调拨时的单位零售价快照
  price_per_unit numeric(10,2) NOT NULL DEFAULT 0,

  -- Transfer quantity: must be > 0; serialized items must be 1
  -- 调拨数量：必须大于 0；serialized 商品必须为 1
  qty integer NOT NULL,

  -- Serial number text: only for serialized items (copied from store_serialized_list)
  -- 序列号文本：仅 serialized 商品有值（从 store_serialized_list 复制）
  serial text DEFAULT NULL,

  -- CHECK: quantity must be positive (no zero or negative transfers)
  -- 约束：数量必须为正数（调拨不允许 0 或负数）
  CONSTRAINT chk_store_transfer_line_list_qty_positive
    CHECK (qty > 0)
);

-- =============================================
-- Trigger: validate inventory_mode consistency for transfer lines
-- 触发器：校验调拨明细行的 inventory_mode 一致性
-- =============================================
-- Rules:
--   service    → reject (service items have no inventory to transfer)
--   serialized → qty must be 1, unit_id and serial must be set
--   tracked / untracked → unit_id and serial must be NULL
-- 规则：
--   service    → 拒绝（服务类无库存，无需调拨）
--   serialized → qty 必须为 1，unit_id 和 serial 必须有值
--   tracked / untracked → unit_id 和 serial 必须为 NULL
-- =============================================
CREATE OR REPLACE FUNCTION public.transfer_line_enforce_mode()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  -- Variable legend / 变量说明:
  --   v_mode = inventory_mode from mother table / 母表的库存跟踪模式
  v_mode public.inventory_mode;
BEGIN
  SELECT inventory_mode
    INTO v_mode
  FROM public.mother_inventory_list
  WHERE unique_id = NEW.unique_id;

  IF NOT FOUND THEN
    RAISE EXCEPTION 'mother_inventory_list.unique_id % not found', NEW.unique_id;
  END IF;

  -- Service items cannot be transferred (no inventory to move)
  -- service 商品不可调拨（无库存可移动）
  IF v_mode = 'service' THEN
    RAISE EXCEPTION 'service item (unique_id=%) cannot be transferred', NEW.unique_id;
  END IF;

  -- Serialized items: qty must be 1, must have unit_id and serial
  -- serialized 商品：qty 必须为 1，必须有 unit_id 和 serial
  IF v_mode = 'serialized' THEN
    IF NEW.qty != 1 THEN
      RAISE EXCEPTION 'serialized item (unique_id=%) must have qty=1', NEW.unique_id;
    END IF;
    IF NEW.unit_id IS NULL THEN
      RAISE EXCEPTION 'serialized item (unique_id=%) requires unit_id', NEW.unique_id;
    END IF;
    IF NEW.serial IS NULL THEN
      RAISE EXCEPTION 'serialized item (unique_id=%) requires serial', NEW.unique_id;
    END IF;
  END IF;

  -- Tracked / untracked items: must NOT have unit_id or serial
  -- tracked / untracked 商品：不应有 unit_id 和 serial
  IF v_mode IN ('tracked', 'untracked') THEN
    IF NEW.unit_id IS NOT NULL THEN
      RAISE EXCEPTION 'non-serialized item (unique_id=%) must not have unit_id', NEW.unique_id;
    END IF;
    IF NEW.serial IS NOT NULL THEN
      RAISE EXCEPTION 'non-serialized item (unique_id=%) must not have serial', NEW.unique_id;
    END IF;
  END IF;

  RETURN NEW;
END;
$$;

DROP TRIGGER IF EXISTS trg_transfer_line_enforce_mode ON public.store_transfer_line_list;
CREATE TRIGGER trg_transfer_line_enforce_mode
BEFORE INSERT OR UPDATE ON public.store_transfer_line_list
FOR EACH ROW
EXECUTE FUNCTION public.transfer_line_enforce_mode();

-- =============================================
-- Indexes / 索引
-- =============================================

-- Look up all line items for a given transfer
-- 查询某次调拨的所有明细行
CREATE INDEX IF NOT EXISTS idx_transfer_line_transfer
  ON public.store_transfer_line_list (store_transfer_id);

-- Look up transfer history for a specific product
-- 查询某商品的调拨历史
CREATE INDEX IF NOT EXISTS idx_transfer_line_unique_id
  ON public.store_transfer_line_list (unique_id);

-- Look up transfer history for a specific serialized unit
-- 查询某序列号单件的调拨记录
CREATE INDEX IF NOT EXISTS idx_transfer_line_unit_id
  ON public.store_transfer_line_list (unit_id)
  WHERE unit_id IS NOT NULL;


# 表19，store_item_history_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 19 · store_item_history_list — Aggregated inventory change history (IMMUTABLE)
-- 文件 19 · store_item_history_list — 商品库存变动历史汇总表（不可变）
-- =============================================
-- Dependencies / 依赖:
--   01_store_list            (store_id FK)
--   03_user_list             (user_id FK)
--   06_mother_inventory_list (unique_id FK)
-- Dependents / 被依赖:
--   (none) （无）
-- =============================================
-- Aggregated, denormalized snapshot of inventory changes from multiple sources:
-- transactions, adjustments, transfers, serialized events, purchase orders, etc.
-- Used by the client for fast querying and display of product history.
-- Online-only: the client writes to this table during sync (no offline operation).
-- IMMUTABLE: triggers block all UPDATE and DELETE operations.
-- ─────────────────────────────────────────────
-- 汇总来自多个来源的库存变动记录（交易、盘点、调拨、serialized 事件、采购等）。
-- 表内数据为冗余快照，仅用于客户端快速查询和展示商品历史。
-- 不支持离线：由客户端在联网时从其他表同步写入。
-- 不可变：触发器强制拦截所有 UPDATE 和 DELETE 操作。
-- =============================================

CREATE TABLE IF NOT EXISTS public.store_item_history_list (

  -- Auto-increment primary key, starting from 0 (assigned by DB)
  -- 全局自增主键，从 0 开始（由数据库分配）
  history_id integer
    GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0)
    PRIMARY KEY,

  -- Store where the inventory change occurred, FK to store_list
  -- 库存变动发生的门店，外键指向 store_list
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Product reference, FK to mother_inventory_list
  -- 商品引用，外键指向 mother_inventory_list
  unique_id integer NOT NULL REFERENCES public.mother_inventory_list(unique_id),

  -- Inventory quantity change (integer)
  -- Applies to all inventory_mode types:
  --   tracked    = actual qty change (positive = stock in, negative = stock out)
  --   serialized = per-unit change (+1 = received, -1 = sold, etc.)
  --   untracked  = qty from transaction; 0 for adjustments (bucket changes have no numeric qty)
  -- May be 0 (e.g., stocktake confirming no change, untracked bucket adjustment)
  -- 库存数量变动（整数）
  -- 适用于所有 inventory_mode：
  --   tracked    = 实际数量变化（正数入库，负数出库）
  --   serialized = 单件变动（+1 入库，-1 卖出等）
  --   untracked  = 交易时有数量；盘点时统一填 0（档位变更无数值意义）
  -- 允许为 0（如盘点确认无变化、untracked 盘点档位调整等）
  qty_delta integer NOT NULL DEFAULT 0,

  -- Post-event inventory snapshot (TEXT type for flexibility across modes)
  --   tracked    = current quantity as string (e.g., "42")
  --   serialized = count of in_stock units for this SKU in this store (e.g., "5")
  --   untracked  = current stock_bucket label (e.g., "normal", "few")
  -- Display-only; TEXT type accommodates all formats
  -- 事件后的库存快照（TEXT 类型，兼容所有模式的格式）
  --   tracked    = 当前库存数量字符串（如 "42"）
  --   serialized = 该 SKU 在该门店剩余的 in_stock 单件总数（如 "5"）
  --   untracked  = 当前档位描述（如 "normal"、"few"）
  -- 纯展示用途，TEXT 类型兼容所有格式
  qty_snapshot text NOT NULL,

  -- Employee who performed the operation, FK to user_list
  -- 执行操作的员工，外键指向 user_list
  user_id integer NOT NULL REFERENCES public.user_list(user_id),

  -- Event timestamp (NOT auto-generated; read from the source table):
  --   Transaction      → transaction's created_at
  --   Adjustment       → adjustment's created_at
  --   Transfer         → transfer's created_at
  --   Serialized event → serialized_event's created_at
  --   Purchase order   → PO's created_at
  -- 事件发生时间（非自动生成；从来源表读取）：
  --   交易     → 交易的 created_at
  --   盘点     → 盘点的 created_at
  --   调拨     → 调拨的 created_at
  --   序列号事件 → 序列号事件的 created_at
  --   采购     → 采购单的 created_at
  created_at timestamptz NOT NULL,

  -- Event type string (provided by client)
  -- For serialized items: copied from serialized_event_list.event_type
  --   e.g., 'purchase', 'sell', 'return', 'store_transferred', 'repair_out'
  -- For non-serialized items: set by client based on source
  --   e.g., 'sale', 'refund', 'exchange', 'inventory_adjustment',
  --         'store_transfer', 'purchase_order'
  -- 事件类型字符串（由客户端填写）
  -- serialized 商品：从 serialized_event_list.event_type 复制
  --   如 'purchase', 'sell', 'return', 'store_transferred', 'repair_out'
  -- 非 serialized 商品：按来源填写
  --   如 'sale', 'refund', 'exchange', 'inventory_adjustment',
  --      'store_transfer', 'purchase_order'
  event_type text NOT NULL,

  -- Source record ID (provided by client)
  -- Points to the PK of the originating table; TEXT type to accommodate both uuid and integer PKs
  -- e.g., transaction_line_list.line_id, adjustment_line_list.line_id,
  --       transfer_line_list.line_id, serialized_event_list.event_id
  -- 来源记录 ID（由客户端填写）
  -- 指向原始表的主键；TEXT 类型兼容 uuid 和 integer 主键
  -- 如 transaction_line_list.line_id、adjustment_line_list.line_id、
  --    transfer_line_list.line_id、serialized_event_list.event_id
  source_id text NOT NULL
);

-- =============================================
-- Immutability triggers: block all UPDATE and DELETE operations
-- 不可变触发器：拦截所有 UPDATE 和 DELETE 操作
-- =============================================
-- Once a history record is created, it must never be modified or removed.
-- This ensures the audit trail remains intact.
-- 一旦历史记录创建，就不能被修改或删除。
-- 这确保了审计线索的完整性。
-- =============================================
CREATE OR REPLACE FUNCTION public.store_item_history_immutable()
RETURNS trigger
LANGUAGE plpgsql
AS $$
BEGIN
  RAISE EXCEPTION 'store_item_history_list is immutable: % is not allowed', TG_OP;
  RETURN NULL;
END;
$$;

-- Block UPDATE / 拦截 UPDATE
DROP TRIGGER IF EXISTS trg_store_item_history_no_update ON public.store_item_history_list;
CREATE TRIGGER trg_store_item_history_no_update
BEFORE UPDATE ON public.store_item_history_list
FOR EACH ROW
EXECUTE FUNCTION public.store_item_history_immutable();

-- Block DELETE / 拦截 DELETE
DROP TRIGGER IF EXISTS trg_store_item_history_no_delete ON public.store_item_history_list;
CREATE TRIGGER trg_store_item_history_no_delete
BEFORE DELETE ON public.store_item_history_list
FOR EACH ROW
EXECUTE FUNCTION public.store_item_history_immutable();

-- =============================================
-- Indexes / 索引
-- =============================================

-- Core query: complete history for a specific product in a specific store (newest first)
-- 核心查询：某门店某商品的完整历史（按时间倒序）
CREATE INDEX IF NOT EXISTS idx_item_history_store_item
  ON public.store_item_history_list (store_id, unique_id, created_at DESC);

-- All inventory changes in a store (newest first)
-- 门店的所有库存变动（按时间倒序）
CREATE INDEX IF NOT EXISTS idx_item_history_store
  ON public.store_item_history_list (store_id, created_at DESC);

-- All changes for a product across all stores (newest first)
-- 某商品在所有门店的变动历史（按时间倒序）
CREATE INDEX IF NOT EXISTS idx_item_history_unique_id
  ON public.store_item_history_list (unique_id, created_at DESC);

-- Filter by event type (e.g., find all 'sell' events, all 'inventory_adjustment' events)
-- 按事件类型筛选（如查找所有 'sell' 事件、所有 'inventory_adjustment' 事件）
CREATE INDEX IF NOT EXISTS idx_item_history_event_type
  ON public.store_item_history_list (event_type);


# 表20，purchase_order_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =========================================
-- 20 · purchase_order_list 采购订单主表
-- =========================================
-- 依赖: 01_store_list (store_id FK)
--        03_user_list (user_id FK)
--        05_supplier_list (supplier_id FK)
-- 被依赖: 21_purchase_order_line_list (store_id, purchase_order_id 复合 FK)
--
-- 记录从供应商采购商品的订单，仅在联网时可操作，不支持离线

-- =========================================
-- 函数: assign_purchase_order_id_per_store()
-- =========================================
-- 同一 store_id 下，purchase_order_id 从 0 开始依次递增
CREATE OR REPLACE FUNCTION public.assign_purchase_order_id_per_store()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  next_id int;
BEGIN
  IF NEW.store_id IS NULL THEN
    RAISE EXCEPTION 'store_id cannot be null';
  END IF;

  -- 咨询锁：命名空间 1004，避免和其他锁冲突
  PERFORM pg_advisory_xact_lock(1004, hashtext(NEW.store_id));

  SELECT COALESCE(MAX(purchase_order_id), -1) + 1
    INTO next_id
  FROM public.purchase_order_list
  WHERE store_id = NEW.store_id;

  NEW.purchase_order_id := next_id;
  RETURN NEW;
END;
$$;

-- =========================================
-- 建表
-- =========================================
CREATE TABLE IF NOT EXISTS public.purchase_order_list (

  -- 门店 ID
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- 采购单编号，同一门店内从 0 开始自动递增，由触发器分配
  purchase_order_id integer NOT NULL,

  -- 供应商
  supplier_id integer DEFAULT NULL REFERENCES public.supplier_list(supplier_id),

  -- 操作人
  user_id integer NOT NULL REFERENCES public.user_list(user_id),

  -- 采购单备注 / 描述
  description text DEFAULT NULL,

  -- 采购总成本（客户端计算后传入）
  total_cost numeric(10,2) NOT NULL DEFAULT 0,

  created_at timestamptz NOT NULL DEFAULT now(),
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- 软删除时间戳
  deleted_at timestamptz DEFAULT NULL,

  -- 联合主键：同一门店内采购单编号唯一
  CONSTRAINT purchase_order_list_pk PRIMARY KEY (store_id, purchase_order_id)
);

-- =========================================
-- purchase_order_id 自动分配触发器
-- =========================================
DROP TRIGGER IF EXISTS trg_purchase_order_assign_id ON public.purchase_order_list;
CREATE TRIGGER trg_purchase_order_assign_id
BEFORE INSERT ON public.purchase_order_list
FOR EACH ROW
EXECUTE FUNCTION public.assign_purchase_order_id_per_store();

-- =========================================
-- updated_at 自动刷新触发器
-- （复用 03_user_list 中创建的 public.set_updated_at()）
-- =========================================
DROP TRIGGER IF EXISTS trg_purchase_order_updated_at ON public.purchase_order_list;
CREATE TRIGGER trg_purchase_order_updated_at
BEFORE UPDATE ON public.purchase_order_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- =========================================
-- 索引
-- =========================================

-- 某门店的采购单列表（按时间倒序）
CREATE INDEX IF NOT EXISTS idx_purchase_order_store
  ON public.purchase_order_list (store_id, created_at DESC)
  WHERE deleted_at IS NULL;

-- 按供应商查采购单
CREATE INDEX IF NOT EXISTS idx_purchase_order_supplier
  ON public.purchase_order_list (supplier_id)
  WHERE supplier_id IS NOT NULL AND deleted_at IS NULL;


# 表21，purchase_order_line_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =========================================
-- 21 · purchase_order_line_list 采购订单明细表
-- =========================================
-- 依赖: 06_mother_inventory_list (unique_id FK)
--        06 中的 ENUM inventory_mode
--        09_store_serialized_list (unit_id FK)
--        20_purchase_order_list (store_id, purchase_order_id 复合 FK)
-- 被依赖: 无

CREATE TABLE IF NOT EXISTS public.purchase_order_line_list (

  -- 全局自增主键，从 0 开始
  purchase_order_line_id integer
    GENERATED ALWAYS AS IDENTITY (START WITH 0 MINVALUE 0)
    PRIMARY KEY,

  -- 门店 ID（用于复合外键指向主表）
  store_id text NOT NULL,

  -- 所属采购单
  purchase_order_id integer NOT NULL,

  -- 关联母表商品
  unique_id integer NOT NULL REFERENCES public.mother_inventory_list(unique_id),

  -- 序列号商品关联：serialized 商品应有值，非 serialized 商品应为 NULL
  unit_id integer DEFAULT NULL REFERENCES public.store_serialized_list(unit_id),

  -- 商品名称快照
  item_name text NOT NULL,

  -- 单位采购成本
  unit_cost numeric(10,2) NOT NULL DEFAULT 0,

  -- 采购数量：serialized 商品应为 1
  qty integer NOT NULL,

  -- 序列号文本：仅 serialized 商品有值
  serial text DEFAULT NULL,

  -- 备注
  note text DEFAULT NULL,

  created_at timestamptz NOT NULL DEFAULT now(),
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- 软删除时间戳
  deleted_at timestamptz DEFAULT NULL,

  -- 复合外键：指向 purchase_order_list(store_id, purchase_order_id)
  CONSTRAINT purchase_order_line_fk_order
    FOREIGN KEY (store_id, purchase_order_id)
    REFERENCES public.purchase_order_list (store_id, purchase_order_id),

  -- 数量应大于 0
  CONSTRAINT purchase_order_line_qty_positive
    CHECK (qty > 0)
);

-- =========================================
-- Trigger：校验 inventory_mode 一致性
-- =========================================
-- - service 商品：拒绝（service 不跟踪库存，无需采购）
-- - serialized 商品：qty 应为 1，unit_id 和 serial 应有值
-- - tracked / untracked 商品：unit_id 和 serial 应为 NULL
CREATE OR REPLACE FUNCTION public.purchase_order_line_enforce_mode()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  v_mode public.inventory_mode;
BEGIN
  SELECT inventory_mode
    INTO v_mode
  FROM public.mother_inventory_list
  WHERE unique_id = NEW.unique_id;

  IF NOT FOUND THEN
    RAISE EXCEPTION 'mother_inventory_list.unique_id % not found', NEW.unique_id;
  END IF;

  -- service 商品不可采购
  IF v_mode = 'service' THEN
    RAISE EXCEPTION 'service item (unique_id=%) cannot be purchased', NEW.unique_id;
  END IF;

  -- serialized 商品：qty 应为 1，且应有 unit_id 和 serial
  IF v_mode = 'serialized' THEN
    IF NEW.qty != 1 THEN
      RAISE EXCEPTION 'serialized item (unique_id=%) must have qty=1', NEW.unique_id;
    END IF;
    IF NEW.unit_id IS NULL THEN
      RAISE EXCEPTION 'serialized item (unique_id=%) requires unit_id', NEW.unique_id;
    END IF;
    IF NEW.serial IS NULL THEN
      RAISE EXCEPTION 'serialized item (unique_id=%) requires serial', NEW.unique_id;
    END IF;
  END IF;

  -- tracked / untracked 商品：不应有 unit_id 和 serial
  IF v_mode IN ('tracked', 'untracked') THEN
    IF NEW.unit_id IS NOT NULL THEN
      RAISE EXCEPTION 'non-serialized item (unique_id=%) must not have unit_id', NEW.unique_id;
    END IF;
    IF NEW.serial IS NOT NULL THEN
      RAISE EXCEPTION 'non-serialized item (unique_id=%) must not have serial', NEW.unique_id;
    END IF;
  END IF;

  RETURN NEW;
END;
$$;

DROP TRIGGER IF EXISTS trg_purchase_order_line_enforce_mode ON public.purchase_order_line_list;
CREATE TRIGGER trg_purchase_order_line_enforce_mode
BEFORE INSERT OR UPDATE ON public.purchase_order_line_list
FOR EACH ROW
EXECUTE FUNCTION public.purchase_order_line_enforce_mode();

-- =========================================
-- updated_at 自动刷新触发器
-- （复用 03_user_list 中创建的 public.set_updated_at()）
-- =========================================
DROP TRIGGER IF EXISTS trg_purchase_order_line_updated_at ON public.purchase_order_line_list;
CREATE TRIGGER trg_purchase_order_line_updated_at
BEFORE UPDATE ON public.purchase_order_line_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- =========================================
-- 索引
-- =========================================

-- 查某次采购单的所有明细行
CREATE INDEX IF NOT EXISTS idx_purchase_order_line_order
  ON public.purchase_order_line_list (store_id, purchase_order_id)
  WHERE deleted_at IS NULL;

-- 查某商品的采购历史
CREATE INDEX IF NOT EXISTS idx_purchase_order_line_unique_id
  ON public.purchase_order_line_list (unique_id)
  WHERE deleted_at IS NULL;

-- 查序列号单件的采购记录
CREATE INDEX IF NOT EXISTS idx_purchase_order_line_unit_id
  ON public.purchase_order_line_list (unit_id)
  WHERE unit_id IS NOT NULL AND deleted_at IS NULL;


# 表22，repair_ticket_list

```sql
-- Async requirement: YES - offline POS must continue high-frequency essential operations using local snapshot; sync changes to cloud after reconnection.
-- 异步需求：是 - POS 离线时需依赖本地快照继续高频必要操作，网络恢复后将变更同步到云端。
-- =========================================
-- 22 · repair_ticket_list 修理工单主表
-- =========================================
-- 依赖: 01_store_list (store_id FK)
--        03_user_list (user_id / tech_id FK)
--        07_customer_list (customer_id FK)
-- 被依赖: 23_repair_ticket_line_list (repair_ticket_id FK)
--         12_transaction_list (repair_ticket_id FK，本文件通过 ALTER TABLE 添加)
--
-- 记录客户送修设备的工单信息，内容离线运行

-- =========================================
-- ENUM: repair_status 修理工单状态枚举
-- =========================================
--   pending   = 待修理
--   completed = 修理完成
--   paid      = 已付款
--   cancelled = 已取消
DO $$
BEGIN
  CREATE TYPE public.repair_status AS ENUM ('pending', 'completed', 'paid', 'cancelled');
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =========================================
-- 建表
-- =========================================
CREATE TABLE IF NOT EXISTS public.repair_ticket_list (

  -- 客户端生成 UUIDv7，INSERT 时传入
  repair_ticket_id uuid PRIMARY KEY,

  -- 门店
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Human-readable repair document number
  -- Format: {store_code}{device_no}R-{YYMMDD}-{NNN} (e.g. D2R-260303-003)
  -- NULL allowed for legacy rows / offline temp rows before server normalization
  -- 可读修理单号
  -- 格式：{store_code}{device_no}R-{YYMMDD}-{NNN}（例如 D2R-260303-003）
  -- 兼容历史数据与离线临时记录：允许为 NULL
  display_no text,

  -- 旧的整型展示编号，保留用于兼容
  repair_display_id int4,

  -- 客户
  customer_id uuid REFERENCES public.customer_list(customer_id),
  customer_name text,                             -- 快照

  -- 操作人（创建工单的员工）
  user_id int4 REFERENCES public.user_list(user_id),

  -- 负责修理的技师，创建时可能未分配
  tech_id int4 REFERENCES public.user_list(user_id),

  -- 设备信息
  device_name text,
  serial text,                                    -- 客户设备序列号，可选
  condition_before text,                          -- 修理前状况描述
  password_note text,                             -- 客户端加密后存储

  -- 备注
  note_invoice text,                              -- 出现在 invoice 上
  note_store text,                                -- 店内 / 技师查看

  -- 状态
  repair_status public.repair_status NOT NULL DEFAULT 'pending',
  completed_at timestamptz,                       -- 完成时间

  created_at timestamptz NOT NULL DEFAULT now(),
  updated_at timestamptz NOT NULL DEFAULT now(),
  deleted_at timestamptz,
  synced_at timestamptz,

  -- 显示单号格式校验（可空）
  CONSTRAINT chk_repair_ticket_display_no_format
    CHECK (
      display_no IS NULL
      OR display_no ~ '^[A-Za-z0-9]+R-[0-9]{6}-[0-9]{3}$'
    )
);

-- =========================================
-- 索引
-- =========================================

-- 某门店工单列表（按时间倒序）
CREATE INDEX IF NOT EXISTS idx_repair_ticket_store_created
  ON public.repair_ticket_list (store_id, created_at DESC)
  WHERE deleted_at IS NULL;

-- 某客户的工单历史
CREATE INDEX IF NOT EXISTS idx_repair_ticket_customer_created
  ON public.repair_ticket_list (customer_id, created_at DESC)
  WHERE deleted_at IS NULL;

-- 按状态筛选
CREATE INDEX IF NOT EXISTS idx_repair_ticket_status
  ON public.repair_ticket_list (store_id, repair_status)
  WHERE deleted_at IS NULL;

-- display_no 唯一（同门店内，非空且未删除）
CREATE UNIQUE INDEX IF NOT EXISTS uq_repair_ticket_store_display_no
  ON public.repair_ticket_list (store_id, display_no)
  WHERE display_no IS NOT NULL AND deleted_at IS NULL;

-- 旧展示编号唯一（同门店内，非空且未删除）
CREATE UNIQUE INDEX IF NOT EXISTS uq_repair_ticket_store_display_id
  ON public.repair_ticket_list (store_id, repair_display_id)
  WHERE repair_display_id IS NOT NULL AND deleted_at IS NULL;

-- =========================================
-- 迁移兼容补丁：为已存在数据库补充 display_no 列与约束
-- =========================================
ALTER TABLE public.repair_ticket_list
  ADD COLUMN IF NOT EXISTS display_no text;

DO $$
BEGIN
  ALTER TABLE public.repair_ticket_list
    ADD CONSTRAINT chk_repair_ticket_display_no_format
    CHECK (
      display_no IS NULL
      OR display_no ~ '^[A-Za-z0-9]+R-[0-9]{6}-[0-9]{3}$'
    );
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =========================================
-- updated_at 自动刷新触发器
-- （复用 03_user_list 中创建的 public.set_updated_at()）
-- =========================================
DROP TRIGGER IF EXISTS trg_repair_ticket_set_updated_at ON public.repair_ticket_list;
CREATE TRIGGER trg_repair_ticket_set_updated_at
BEFORE UPDATE ON public.repair_ticket_list
FOR EACH ROW EXECUTE FUNCTION public.set_updated_at();

-- =========================================
-- 补丁：为 transaction_list 添加 repair_ticket_id FK
-- =========================================
-- 修理完成后付款时，transaction_list 中的交易通过此字段关联修理工单
ALTER TABLE public.transaction_list
  ADD COLUMN IF NOT EXISTS repair_ticket_id uuid;

DO $$
BEGIN
  ALTER TABLE public.transaction_list
    ADD CONSTRAINT fk_transaction_repair_ticket
    FOREIGN KEY (repair_ticket_id) REFERENCES public.repair_ticket_list(repair_ticket_id);
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;


# 表23，repair_ticket_line_list

```sql
-- Async requirement: YES - offline POS must continue high-frequency essential operations using local snapshot; sync changes to cloud after reconnection.
-- 异步需求：是 - POS 离线时需依赖本地快照继续高频必要操作，网络恢复后将变更同步到云端。
-- =========================================
-- 23 · repair_ticket_line_list 修理工单明细表
-- =========================================
-- 依赖: 01_store_list (store_id FK)
--        06_mother_inventory_list (unique_id FK)
--        22_repair_ticket_list (repair_ticket_id FK)
-- 被依赖: 无
--
-- 每行代表一个修理项目 / issue

CREATE TABLE IF NOT EXISTS public.repair_ticket_line_list (

  -- 客户端生成 UUIDv7
  repair_ticket_line_id uuid PRIMARY KEY,

  -- 所属工单
  repair_ticket_id uuid NOT NULL REFERENCES public.repair_ticket_list(repair_ticket_id),

  -- 门店（冗余，方便离线同步按门店筛选）
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- 关联母表商品（可选，手动录入项目时为 NULL）
  unique_id int4 REFERENCES public.mother_inventory_list(unique_id),

  -- 项目名称（快照 / 手动录入）
  item_name text NOT NULL,

  -- 数量
  qty int4 NOT NULL DEFAULT 1 CHECK (qty > 0),

  -- 成本与售价，允许后续补填
  unit_cost numeric(10,2),
  unit_price numeric(10,2),

  -- 备注
  note text,

  created_at timestamptz NOT NULL DEFAULT now(),
  updated_at timestamptz NOT NULL DEFAULT now(),
  deleted_at timestamptz,
  synced_at timestamptz
);

-- =========================================
-- 索引
-- =========================================

-- 查某工单的所有明细行
CREATE INDEX IF NOT EXISTS idx_repair_ticket_line_ticket
  ON public.repair_ticket_line_list (repair_ticket_id)
  WHERE deleted_at IS NULL;

-- 查某商品的修理使用历史
CREATE INDEX IF NOT EXISTS idx_repair_ticket_line_unique_id
  ON public.repair_ticket_line_list (unique_id)
  WHERE unique_id IS NOT NULL AND deleted_at IS NULL;

-- =========================================
-- updated_at 自动刷新触发器
-- （复用 03_user_list 中创建的 public.set_updated_at()）
-- =========================================
DROP TRIGGER IF EXISTS trg_repair_ticket_line_set_updated_at ON public.repair_ticket_line_list;
CREATE TRIGGER trg_repair_ticket_line_set_updated_at
BEFORE UPDATE ON public.repair_ticket_line_list
FOR EACH ROW EXECUTE FUNCTION public.set_updated_at();


# 表24，store_demand_list

```sql
-- Async requirement: NO - cloud-first table; offline local snapshot support is not required for high-frequency essential POS operations.
-- 异步需求：否 - 该表采用云端优先，不要求离线本地快照支持高频必要 POS 操作。
-- =============================================
-- File 24 · store_demand_list — Store demand quick-capture table
-- 文件 24 · store_demand_list — 门店需求快速记录表
-- =============================================
-- Dependencies / 依赖:
--   01_store_list (store_id FK)
--   03_user_list  (created_by / reviewed_by FK)
-- Dependents / 被依赖:
--   (none)
-- Shared components created here / 本文件创建的共享组件:
--   ENUM public.demand_status
--   Function public.assign_demand_id_per_store()
-- =============================================
-- Purpose:
--   Lightweight shared table for quickly recording daily demands/requests.
--   One row can contain one or multiple demand items in plain text.
-- ─────────────────────────────────────────────
-- 目的：
--   提供给全员快速记录需求的轻量表。
--   一行可记录一个或多个需求项（纯文本）。
-- =============================================

-- =============================================
-- ENUM: demand_status — demand review workflow state
-- 枚举：demand_status — 需求审核流程状态
-- =============================================
DO $$
BEGIN
  CREATE TYPE public.demand_status AS ENUM (
    'pending', 'processing', 'rejected', 'done'
  );
EXCEPTION
  WHEN duplicate_object THEN NULL;
END $$;

-- =============================================
-- Function: assign_demand_id_per_store()
-- 函数：assign_demand_id_per_store()
-- =============================================
-- Auto-assigns demand_id per store starting from 0.
-- Uses advisory lock namespace 1010 to avoid concurrent conflicts.
-- ─────────────────────────────────────────────
-- 按门店自动分配 demand_id，从 0 开始递增。
-- 使用咨询锁命名空间 1010 防止并发冲突。
-- =============================================
CREATE OR REPLACE FUNCTION public.assign_demand_id_per_store()
RETURNS trigger
LANGUAGE plpgsql
AS $$
DECLARE
  next_demand_id integer;
BEGIN
  IF NEW.store_id IS NULL THEN
    RAISE EXCEPTION 'store_id cannot be null';
  END IF;

  PERFORM pg_advisory_xact_lock(1010, hashtext(NEW.store_id));

  SELECT COALESCE(MAX(demand_id), -1) + 1
    INTO next_demand_id
  FROM public.store_demand_list
  WHERE store_id = NEW.store_id;

  NEW.demand_id := next_demand_id;
  RETURN NEW;
END;
$$;

-- =============================================
-- Table creation / 建表
-- =============================================
CREATE TABLE IF NOT EXISTS public.store_demand_list (

  -- Store ID, FK to store_list
  -- 门店 ID，外键指向 store_list
  store_id text NOT NULL REFERENCES public.store_list(store_id),

  -- Per-store demand number, auto-assigned by trigger (starts at 0)
  -- 门店内需求编号，由触发器自动分配（从 0 开始）
  demand_id integer NOT NULL,

  -- Demand tag/category (client-managed vocabulary)
  -- 需求标签/分类（由客户端管理标签字典）
  tag text NOT NULL,

  -- Review workflow status
  -- 审核流程状态
  status public.demand_status NOT NULL DEFAULT 'pending',

  -- Demand content in plain text; can contain multiple items
  -- 需求正文（纯文本）；可在一行中写多个需求项
  content text NOT NULL,

  -- Creator user
  -- 创建人
  created_by integer NOT NULL REFERENCES public.user_list(user_id),

  -- Creation time
  -- 创建时间
  created_at timestamptz NOT NULL DEFAULT now(),

  -- Reviewer user (manager+), NULL before review
  -- 审核人（manager+），未审核前为 NULL
  reviewed_by integer DEFAULT NULL REFERENCES public.user_list(user_id),

  -- Review timestamp, NULL before review
  -- 审核时间，未审核前为 NULL
  reviewed_at timestamptz DEFAULT NULL,

  -- Review note / rejection reason / handling instruction
  -- 审核备注 / 驳回原因 / 处理说明
  manager_note text DEFAULT NULL,

  -- Last update timestamp (auto-refreshed by trigger)
  -- 最后更新时间（由触发器自动刷新）
  updated_at timestamptz NOT NULL DEFAULT now(),

  -- Soft delete timestamp
  -- 软删除时间戳
  deleted_at timestamptz DEFAULT NULL,

  -- Composite PK ensures demand_id uniqueness within each store
  -- 联合主键保证 demand_id 在门店内唯一
  CONSTRAINT store_demand_list_pkey PRIMARY KEY (store_id, demand_id),

  -- demand_id must be non-negative
  -- demand_id 必须为非负整数
  CONSTRAINT chk_store_demand_id_non_negative CHECK (demand_id >= 0),

  -- Review fields should be set together (both NULL or both non-NULL)
  -- 审核字段应同时为空或同时有值
  CONSTRAINT chk_store_demand_review_pair CHECK (
    (reviewed_by IS NULL AND reviewed_at IS NULL)
    OR
    (reviewed_by IS NOT NULL AND reviewed_at IS NOT NULL)
  )
);

-- =============================================
-- Trigger: auto-assign demand_id on INSERT
-- 触发器：INSERT 时自动分配 demand_id
-- =============================================
DROP TRIGGER IF EXISTS trg_store_demand_assign_id ON public.store_demand_list;
CREATE TRIGGER trg_store_demand_assign_id
BEFORE INSERT ON public.store_demand_list
FOR EACH ROW
EXECUTE FUNCTION public.assign_demand_id_per_store();

-- =============================================
-- Trigger: auto-refresh updated_at
-- 触发器：自动刷新 updated_at
-- =============================================
DROP TRIGGER IF EXISTS trg_store_demand_set_updated_at ON public.store_demand_list;
CREATE TRIGGER trg_store_demand_set_updated_at
BEFORE UPDATE ON public.store_demand_list
FOR EACH ROW
EXECUTE FUNCTION public.set_updated_at();

-- =============================================
-- Indexes / 索引
-- =============================================

-- Main list query: store demands sorted by newest first (active rows only)
-- 主列表查询：门店需求按时间倒序（仅活跃记录）
CREATE INDEX IF NOT EXISTS idx_store_demand_store_created
  ON public.store_demand_list (store_id, created_at DESC)
  WHERE deleted_at IS NULL;

-- Filter by status within a store
-- 门店内按状态筛选
CREATE INDEX IF NOT EXISTS idx_store_demand_store_status_created
  ON public.store_demand_list (store_id, status, created_at DESC)
  WHERE deleted_at IS NULL;

-- Filter by tag within a store
-- 门店内按标签筛选
CREATE INDEX IF NOT EXISTS idx_store_demand_store_tag_created
  ON public.store_demand_list (store_id, tag, created_at DESC)
  WHERE deleted_at IS NULL;


# wow